# Glaucoma MLDL — one-notebook Colab experiment

**Run cells from top to bottom.** This notebook downloads ACRIMA, RIM-ONE r3, a combined ORIGA/REFUGE archive, the original-source REFUGE archive, and the DRISHTI-GS1 Kaggle mirror. No dataset ZIP or project folder must be uploaded. It writes all training code from cells inside this notebook.

The notebook trains classification and optic-disc/cup segmentation baselines, a pooled fine-tuned classifier, and a matched whole-image versus predicted-disc-crop comparison. It evaluates held-out images, creates Grad-CAM and mask overlays, builds architecture and metric figures, and exports measured results for a later static website.

**Source note:** DRISHTI-GS1 comes from a third-party Kaggle mirror because that is the source chosen for this run. Label its results as *mirror-based* in the report. Do not redistribute its images. The original IIIT Hyderabad terms allow redistribution only within a registered team. REFUGE official test labels come from its original-source RAR; the combined archive's test index alone does not contain them.

**Time and storage:** The four public archives total roughly 10.5 GB, plus the DRISHTI mirror. Download and training can take hours on a free Colab session. Choose a GPU runtime before the training cells. Results depend on actual training; no metric in this notebook is pre-filled.

In [ ]:
# 0 — Environment, folders, and reproducible configuration
import csv, io, json, os, random, shutil, subprocess, sys, zipfile
from pathlib import Path

ROOT = Path('/content/glaucoma_mldl') if Path('/content').exists() else Path.cwd() / 'glaucoma_mldl_run'
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
for folder in ('tools', 'models', 'data/raw', 'data/audit', 'data/provisional/drishti_gs1', 'outputs', 'results', 'website', 'report'):
    Path(folder).mkdir(parents=True, exist_ok=True)

required = {'torch':'torch', 'torchvision':'torchvision', 'numpy':'numpy', 'PIL':'Pillow', 'sklearn':'scikit-learn', 'matplotlib':'matplotlib', 'openpyxl':'openpyxl', 'requests':'requests', 'joblib':'joblib'}
import importlib.util
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
import numpy as np, torch
from PIL import Image
from IPython.display import display

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Working directory:', ROOT)
print('Device:', DEVICE, '| PyTorch:', torch.__version__)
print('Free disk GiB:', round(shutil.disk_usage(ROOT).free / 2**30, 1))
if shutil.disk_usage(ROOT).free < 15 * 2**30:
    raise RuntimeError('At least 15 GiB of free space is needed for the source archives. Free Colab disk space and rerun this cell.')

RUN_DOWNLOADS = True
RUN_BASELINES = True
RUN_POOLED_SEGMENTER = True
RUN_FINETUNING = True
RUN_VISUALS_AND_EXPORT = True
FORCE_RETRAIN = False

def run_script(name, required_output=None):
    target = ROOT / 'tools' / name
    if required_output and (ROOT / required_output).exists() and not FORCE_RETRAIN:
        print('Already complete:', required_output)
        return
    print('RUN:', name, flush=True)
    subprocess.run([sys.executable, str(target)], cwd=ROOT, check=True)

print('Setup complete. Continue to Step 1.')

## Step 1 — Download the five dataset sources

The commands fetch all archives directly. Interrupted downloads resume. `data/raw/` and `data/provisional/` are created in this fresh runtime; no upload is needed.

Run the source cells below, then the stage runner. Every file they create comes from code **inside this notebook**.

In [ ]:
%%writefile tools/download_verified.py
"""Download only source-verified dataset archives, with resumable transfers."""
from __future__ import annotations

import argparse
import hashlib
import json
import time
from pathlib import Path
from urllib.error import URLError
from urllib.request import Request, urlopen

SOURCES = {
    "acrima": ("https://ndownloader.figshare.com/files/14137700", 23608145, "ACRIMA_Database.zip"),
    "rim_one_r3": ("https://medimrg.webs.ull.es/RIM-ONE-r3.zip", 203123809, "RIM-ONE-r3.zip"),
    "refuge_candidate": ("https://ndownloader.figshare.com/files/47095942", 4265372818, "REFUGE_figshare.rar"),
    "origa_candidate": ("https://ndownloader.figshare.com/files/43119880", 5956275477, "ORIGA_candidate_figshare.zip"),
}


def download(name: str, root: Path) -> None:
    url, size, filename = SOURCES[name]
    root.mkdir(parents=True, exist_ok=True)
    target = root / filename
    partial = root / f"{filename}.part"
    if target.exists() and target.stat().st_size == size:
        print(f"{name}: already downloaded", flush=True)
        return
    for attempt in range(1, 8):
        offset = partial.stat().st_size if partial.exists() else 0
        try:
            request = Request(url, headers={"User-Agent": "MLDL-research/0.1", **({"Range": f"bytes={offset}-"} if offset else {})})
            with urlopen(request, timeout=120) as response:
                if offset and response.status != 206:
                    offset = 0
                with partial.open("ab" if offset else "wb") as output:
                    while chunk := response.read(1024 * 1024):
                        output.write(chunk)
            break
        except (ConnectionResetError, TimeoutError, URLError, OSError) as exc:
            if attempt == 7:
                raise
            print(f"{name}: transfer interrupted ({type(exc).__name__}); retrying from byte {partial.stat().st_size if partial.exists() else 0}", flush=True)
            time.sleep(min(2 ** attempt, 30))
    actual = partial.stat().st_size
    if actual != size:
        raise RuntimeError(f"{name}: expected {size} bytes, got {actual}; partial retained")
    partial.replace(target)
    sha = hashlib.sha256()
    with target.open("rb") as source:
        while chunk := source.read(1024 * 1024):
            sha.update(chunk)
    (root / f"{filename}.provenance.json").write_text(json.dumps({"dataset": name, "source_url": url, "bytes": size, "sha256": sha.hexdigest()}, indent=2))
    print(f"{name}: {target} | sha256={sha.hexdigest()}", flush=True)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("dataset", choices=SOURCES)
    parser.add_argument("--output", type=Path, default=Path("data/raw"))
    args = parser.parse_args()
    download(args.dataset, args.output)


In [ ]:
%%writefile tools/download_drishti_kaggle_inventory.py
"""Download the user-selected DRISHTI-GS Kaggle mirror with explicit provenance."""
from __future__ import annotations

import hashlib
import json
import time
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


DATASET = "lokeshsaipureddi/drishtigs-retina-dataset-for-onh-segmentation"
URL = f"https://www.kaggle.com/api/v1/datasets/download/{DATASET}"
OUT = Path("data/provisional/drishti_gs1")
TARGET = OUT / "drishti_gs1_kaggle_mirror.zip"
PARTIAL = OUT / "drishti_gs1_kaggle_mirror.zip.part"


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(1024 * 1024):
            digest.update(chunk)
    return digest.hexdigest()


OUT.mkdir(parents=True, exist_ok=True)
if TARGET.exists():
    print(json.dumps({"state": "complete", "path": str(TARGET), "bytes": TARGET.stat().st_size, "sha256": sha256(TARGET)}, indent=2))
    raise SystemExit

for attempt in range(1, 8):
    offset = PARTIAL.stat().st_size if PARTIAL.exists() else 0
    try:
        headers = {"User-Agent": "MLDL-research/0.1"}
        if offset:
            headers["Range"] = f"bytes={offset}-"
        request = Request(URL, headers=headers)
        with urlopen(request, timeout=180) as response:
            if offset and response.status != 206:
                offset = 0
                PARTIAL.unlink(missing_ok=True)
            with PARTIAL.open("ab" if offset else "wb") as stream:
                while chunk := response.read(1024 * 1024):
                    stream.write(chunk)
        break
    except (HTTPError, URLError, TimeoutError, ConnectionResetError, OSError) as exc:
        if attempt == 7:
            raise
        print(f"Transfer interrupted ({type(exc).__name__}); retained {PARTIAL.stat().st_size if PARTIAL.exists() else 0} bytes", flush=True)
        time.sleep(min(2 ** attempt, 30))

PARTIAL.replace(TARGET)
record = {
    "dataset": "DRISHTI-GS1",
    "source": "Third-party Kaggle mirror",
    "uploader": "lokeshsaipureddi",
    "url": f"https://www.kaggle.com/datasets/{DATASET}",
    "kaggle_metadata_license": "Kaggle page advertises CC0",
    "official_terms": "Official IIIT page limits use to academic/research and prohibits redistribution except within the registered team",
    "mirror_based_experiment": True,
    "official_source_verified": False,
    "bytes": TARGET.stat().st_size,
    "sha256": sha256(TARGET),
}
(OUT / "provenance.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
print(json.dumps(record, indent=2))


In [ ]:
%%writefile tools/verify_figshare_archives.py
"""Compare completed candidate archives with Figshare's published MD5 hashes."""
import hashlib
from pathlib import Path


SOURCES = {
    "REFUGE": (Path("data/raw/REFUGE_figshare.rar"), "963df1526e63c4e437dba37dcd8c89ae"),
    "ORIGA candidate": (Path("data/raw/ORIGA_candidate_figshare.zip"), "8838f860d6d7de42ac00034e3b1ca2df"),
}
for name, (path, expected) in SOURCES.items():
    if not path.exists():
        print(name, "not complete")
        continue
    digest = hashlib.md5()  # Source integrity check only; not a security signature.
    with path.open("rb") as stream:
        while chunk := stream.read(4 * 1024 * 1024):
            digest.update(chunk)
    actual = digest.hexdigest()
    print(name, actual, "matches Figshare" if actual == expected else "MISMATCH")
    assert actual == expected, f"{name} archive checksum differs from Figshare metadata"


In [ ]:
# Download and verify the original source archives plus the chosen DRISHTI mirror.
if RUN_DOWNLOADS:
    for name in ('acrima', 'rim_one_r3', 'origa_candidate', 'refuge_candidate'):
        subprocess.run([sys.executable, 'tools/download_verified.py', name], check=True)
    run_script('download_drishti_kaggle_inventory.py')
    run_script('verify_figshare_archives.py')
else:
    print('Downloads skipped by configuration; checking local runtime files.')
for name in ('ACRIMA_Database.zip', 'RIM-ONE-r3.zip', 'ORIGA_candidate_figshare.zip', 'REFUGE_figshare.rar'):
    assert (ROOT / 'data/raw' / name).is_file(), name
assert (ROOT / 'data/provisional/drishti_gs1/drishti_gs1_kaggle_mirror.zip').is_file()
print('Step 1 complete: five source archives are present.')

## Step 2 — Audit labels, masks, splits, and provenance

The REFUGE RAR needs a RAR-capable extractor. This cell installs the free `bsdtar` utility if Linux does not already have one. Audits stop on missing or mismatched members; the DRISHTI source remains explicitly marked as a mirror.

Run the source cells below, then the stage runner. Every file they create comes from code **inside this notebook**.

In [ ]:
%%writefile tools/audit_acrima.py
"""Validate ACRIMA labels against the archive readme and make a visual audit sheet."""
from __future__ import annotations

import csv
import io
import re
import zipfile
from pathlib import Path
from PIL import Image, ImageDraw

archive = Path("data/raw/ACRIMA_Database.zip")
out = Path("data/audit")
out.mkdir(parents=True, exist_ok=True)
rows = []
with zipfile.ZipFile(archive) as z:
    names = sorted(n for n in z.namelist() if n.lower().endswith(".jpg"))
    for name in names:
        filename = Path(name).name
        match = re.fullmatch(r"Im(\d{3})(_g)?_ACRIMA\.jpg", filename, re.IGNORECASE)
        if not match:
            raise ValueError(f"Unexpected ACRIMA filename: {filename}")
        with Image.open(io.BytesIO(z.read(name))) as img:
            img.verify()
        rows.append({"dataset": "ACRIMA", "archive": str(archive), "member": name, "image_id": match.group(1), "label": "glaucoma" if match.group(2) else "normal", "has_disc_mask": False, "has_cup_mask": False})
    n_g = sum(r["label"] == "glaucoma" for r in rows)
    assert (len(rows), n_g) == (705, 396), (len(rows), n_g)
    chosen = [next(r for r in rows if r["label"] == label) for label in ("normal", "glaucoma")]
    sheet = Image.new("RGB", (800, 430), "#f6f8f9")
    draw = ImageDraw.Draw(sheet)
    for idx, row in enumerate(chosen):
        with Image.open(io.BytesIO(z.read(row["member"]))) as img:
            img.thumbnail((370, 350))
            sheet.paste(img, (20 + idx * 400 + (370 - img.width)//2, 35))
        draw.text((20 + idx * 400, 395), f"{row['label']} | {Path(row['member']).name}", fill="black")
    sheet.save(out / "acrima_samples.jpg", quality=92)
with (out / "acrima_manifest.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
print(f"ACRIMA verified: {len(rows)} images, {n_g} glaucoma, {len(rows)-n_g} normal; no masks")


In [ ]:
%%writefile tools/audit_rim_one_r3.py
"""Audit original RIM-ONE r3 archive labels and paired average masks."""
import collections
import csv
import zipfile
from pathlib import Path

archive = Path("data/raw/RIM-ONE-r3.zip")
out = Path("data/audit")
out.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as z:
    names = set(z.namelist())
    images = sorted(n for n in names if n.lower().endswith(".jpg"))
    rows = []
    for image in images:
        prefix = Path(image).name.split("-")[0]
        assert prefix in {"N", "G", "S"}, image
        base = image.rsplit("/Stereo Images/", 1)[0]
        stem = Path(image).stem
        disc = f"{base}/Average_masks/{stem}-Disc-Avg.png"
        cup = f"{base}/Average_masks/{stem}-Cup-Avg.png"
        assert disc in names and cup in names, (image, disc, cup)
        rows.append({"image": image, "label_original": {"N":"normal","G":"glaucoma","S":"suspect"}[prefix], "disc_mask": disc, "cup_mask": cup})
assert collections.Counter(r["label_original"] for r in rows) == {"normal":85,"glaucoma":39,"suspect":35}
with (out / "rim_one_r3_manifest.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
print("RIM-ONE r3 verified: 85 normal, 39 glaucoma, 35 suspect; all 159 have disc and cup average masks")


In [ ]:
%%writefile tools/audit_origa_refuge.py
"""Audit ORIGA and REFUGE members in the checksum-verified combined archive."""
import csv
import io
import json
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw


archive_path = Path("data/raw/ORIGA_candidate_figshare.zip")
audit_root = Path("data/audit")
audit_root.mkdir(parents=True, exist_ok=True)


def sample_panel(z, image_member, mask_member, destination):
    image = Image.open(io.BytesIO(z.read(image_member))).convert("RGB")
    mask = Image.open(io.BytesIO(z.read(mask_member))).convert("L")
    assert image.size == mask.size, (image_member, mask_member, image.size, mask.size)
    image = image.resize((512, 512))
    mask = mask.resize((512, 512), Image.Resampling.NEAREST)
    values = np.asarray(mask)
    assert set(np.unique(values)) == {0, 1, 2}
    original = np.asarray(image).copy()
    overlay = original.copy()
    overlay[values == 1] = 0.55 * overlay[values == 1] + 0.45 * np.array([0, 220, 110])
    overlay[values == 2] = 0.50 * overlay[values == 2] + 0.50 * np.array([255, 180, 0])
    panel = Image.new("RGB", (1024, 540), "white")
    panel.paste(image, (0, 0))
    panel.paste(Image.fromarray(overlay.astype("uint8")), (512, 0))
    draw = ImageDraw.Draw(panel)
    draw.text((10, 518), "Original", fill="black")
    draw.text((522, 518), "Mask values: 1=outer ring, 2=inner region", fill="black")
    panel.save(destination)


with zipfile.ZipFile(archive_path) as z:
    members = set(z.namelist())
    origa = list(csv.DictReader(io.TextIOWrapper(z.open("ORIGA/OrigaList.csv"), encoding="utf-8-sig")))
    assert len(origa) == 650
    assert len({r["Filename"] for r in origa}) == len(origa)
    assert {r["Glaucoma"] for r in origa} == {"0", "1"}
    assert {r["Set"] for r in origa} == {"A", "B"}
    origa_manifest = []
    for r in origa:
        image = "ORIGA/Images/" + r["Filename"]
        mask = "ORIGA/Masks/" + Path(r["Filename"]).with_suffix(".png").name
        assert image in members and mask in members, (image, mask)
        origa_manifest.append({"image": image, "mask": mask, "label": r["Glaucoma"], "provided_set": r["Set"], "eye": r["Eye"], "expert_cdr": r["ExpCDR"]})
    with (audit_root / "origa_manifest.csv").open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(origa_manifest[0]))
        writer.writeheader()
        writer.writerows(origa_manifest)

    refuge_manifest = []
    refuge_summary = {}
    for split in ("train", "val", "test"):
        index = json.load(z.open(f"REFUGE/{split}/index.json"))
        records = list(index.values())
        assert len(records) == 400
        assert len({r["ImgName"] for r in records}) == len(records)
        labels = Counter(str(r.get("Label", "missing")) for r in records)
        if split == "test":
            assert labels == {"missing": 400}, "Do not infer missing test labels"
        else:
            assert set(labels) == {"0", "1"}
        for r in records:
            stem = Path(r["ImgName"]).stem
            image = f"REFUGE/{split}/Images/{r['ImgName']}"
            mask = f"REFUGE/{split}/Masks/{stem}.png"
            assert image in members and mask in members, (image, mask)
            refuge_manifest.append({"image": image, "mask": mask, "split": split, "label": r.get("Label", "")})
        refuge_summary[split] = {"images": len(records), "labels": dict(labels)}
    with (audit_root / "refuge_manifest.csv").open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(refuge_manifest[0]))
        writer.writeheader()
        writer.writerows(refuge_manifest)

    sample_panel(z, origa_manifest[0]["image"], origa_manifest[0]["mask"], audit_root / "origa_sample.png")
    sample_panel(z, refuge_manifest[0]["image"], refuge_manifest[0]["mask"], audit_root / "refuge_sample.png")

summary = {
    "archive": str(archive_path),
    "source": "Figshare article 24549217, third-party combined archive; MD5 checked separately",
    "origa": {"images": len(origa_manifest), "labels": dict(Counter(r["label"] for r in origa_manifest)), "provided_sets": dict(Counter(r["provided_set"] for r in origa_manifest))},
    "refuge": refuge_summary,
    "mask_encoding_sample": "0 background, 1 outer ring, 2 inner region; verify semantics against source before final training",
}
(audit_root / "origa_refuge_audit.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))


In [ ]:
%%writefile tools/audit_refuge_original.py
"""Recover official REFUGE test labels and check the repackaged archive against RAR samples."""
import csv
import io
import json
import shutil
import subprocess
import shutil
RAR_TAR = shutil.which('bsdtar') or 'tar'
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import openpyxl
from PIL import Image


rar = Path("data/raw/REFUGE_figshare.rar")
combined = Path("data/raw/ORIGA_candidate_figshare.zip")
audit = Path("data/audit")
if shutil.which("7z"):
    listed = subprocess.run(["7z", "l", "-slt", str(rar)], capture_output=True, text=True, check=True).stdout
    members = set()
    for line in listed.splitlines():
        if line.startswith("Path = ") and line != "Path = ":
            member = line.removeprefix("Path = ").strip()
            if member != str(rar):
                members.add(member)
else:
    members = set(subprocess.run([RAR_TAR, "-tf", str(rar)], capture_output=True, text=True, check=True).stdout.splitlines())
label_member = "Refuge/REFUGE-Test-GT/Glaucoma_label_and_Fovea_location.xlsx"
assert label_member in members


def read_rar(member):
    if shutil.which("7z"):
        output = Path("/tmp/refuge_rar_member")
        output.mkdir(parents=True, exist_ok=True)
        for old in output.iterdir():
            old.unlink()
        subprocess.run(["7z", "e", "-y", f"-o{output}", str(rar), member], capture_output=True, check=True)
        files = list(output.iterdir())
        if len(files) != 1:
            raise RuntimeError(f"Expected one extracted file for {member}, got {files}")
        return files[0].read_bytes()
    return subprocess.run([RAR_TAR, "-xOf", str(rar), member], capture_output=True, check=True).stdout


book = openpyxl.load_workbook(io.BytesIO(read_rar(label_member)), read_only=True, data_only=True)
rows = list(book.active.values)
assert rows[0][:3] == ("ID", "ImgName", "Label(Glaucoma=1)")
test_labels = {str(row[1]): int(row[2]) for row in rows[1:]}
assert len(test_labels) == 400 and set(test_labels.values()) == {0, 1}
assert set(test_labels) == {f"T{i:04d}.jpg" for i in range(1, 401)}
with (audit / "refuge_test_labels_official.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(["image", "label"])
    writer.writerows(sorted(test_labels.items()))

manifest_path = audit / "refuge_manifest.csv"
manifest = list(csv.DictReader(manifest_path.open(newline="", encoding="utf-8")))
assert len(manifest) == 1200
for record in manifest:
    if record["split"] == "test":
        record["label"] = test_labels[Path(record["image"]).name]
with manifest_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(manifest[0]))
    writer.writeheader()
    writer.writerows(manifest)

# Check every expected original file name, then compare representative image bytes
# and mask pixels across train, validation, and test.
checks = []
with zipfile.ZipFile(combined) as z:
    combined_members = set(z.namelist())
    for split in ("train", "val", "test"):
        subset = sorted((r for r in manifest if r["split"] == split), key=lambda r: r["image"])
        assert len(subset) == 400
        for record in subset:
            filename = Path(record["image"]).name
            stem = Path(filename).stem
            label = int(record["label"])
            if split == "train":
                cls = "Glaucoma" if label else "Non-Glaucoma"
                original_image = f"Refuge/Training400/{cls}/{filename}"
                original_mask = f"Refuge/Annotation-Training400/Disc_Cup_Masks/{cls}/{stem}.bmp"
            elif split == "val":
                original_image = f"Refuge/REFUGE-Validation400/REFUGE-Validation400/{filename}"
                original_mask = f"Refuge/REFUGE-Validation400-GT/REFUGE-Validation400-GT/Disc_Cup_Masks/{stem}.bmp"
            else:
                original_image = f"Refuge/Test400/{filename}"
                original_mask = f"Refuge/REFUGE-Test-GT/Disc_Cup_Masks/{'G' if label else 'N'}/{stem}.bmp"
            assert original_image in members and original_mask in members, (original_image, original_mask)
            assert record["image"] in combined_members and record["mask"] in combined_members
        for record in (subset[0], subset[199], subset[-1]):
            filename = Path(record["image"]).name
            stem = Path(filename).stem
            label = int(record["label"])
            if split == "train":
                cls = "Glaucoma" if label else "Non-Glaucoma"
                original_image = f"Refuge/Training400/{cls}/{filename}"
                original_mask = f"Refuge/Annotation-Training400/Disc_Cup_Masks/{cls}/{stem}.bmp"
            elif split == "val":
                original_image = f"Refuge/REFUGE-Validation400/REFUGE-Validation400/{filename}"
                original_mask = f"Refuge/REFUGE-Validation400-GT/REFUGE-Validation400-GT/Disc_Cup_Masks/{stem}.bmp"
            else:
                original_image = f"Refuge/Test400/{filename}"
                original_mask = f"Refuge/REFUGE-Test-GT/Disc_Cup_Masks/{'G' if label else 'N'}/{stem}.bmp"
            assert read_rar(original_image) == z.read(record["image"]), (split, filename, "image differs")
            original = np.asarray(Image.open(io.BytesIO(read_rar(original_mask))))
            repackaged = np.asarray(Image.open(io.BytesIO(z.read(record["mask"]))))
            assert original.shape == repackaged.shape
            assert np.array_equal(np.where(original == 255, 0, np.where(original == 128, 1, 2)), repackaged), (split, filename, "mask differs")
            checks.append({"split": split, "image": filename, "image_bytes_identical": True, "mask_pixels_equivalent": True})

summary = {
    "source": "Figshare article 26049574, Refuge.rar, published MD5 verified separately",
    "original_members": len(members),
    "official_test_labels": dict(Counter(test_labels.values())),
    "repackaged_audit": "All 1200 original image/mask names found; 9 sampled images byte-identical and masks pixel-equivalent after value mapping",
    "sample_checks": checks,
}
(audit / "refuge_original_audit.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in summary.items() if k != "sample_checks"}, indent=2))


In [ ]:
%%writefile tools/audit_drishti_mirror.py
"""Create the working DRISHTI-GS1 manifest from the requested Kaggle mirror."""
from __future__ import annotations

import csv
import io
import json
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw


archive = Path("data/provisional/drishti_gs1/drishti_gs1_kaggle_mirror.zip")
audit = Path("data/audit")
audit.mkdir(parents=True, exist_ok=True)
rows = []

with zipfile.ZipFile(archive) as z:
    members = set(z.namelist())
    for name in sorted(n for n in members if n.lower().endswith(".png") and "/images/" in n.lower()):
        lowered = name.lower()
        split = "train" if "/training/" in lowered else "test" if "/test/" in lowered else None
        label = "glaucoma" if "/glaucoma/" in lowered else "normal"
        stem = Path(name).stem
        if split == "train":
            disc = f"Training-20211018T055246Z-001/Training/GT/{stem}/SoftMap/{stem}_ODsegSoftmap.png"
            cup = f"Training-20211018T055246Z-001/Training/GT/{stem}/SoftMap/{stem}_cupsegSoftmap.png"
        else:
            disc = f"Test-20211018T060000Z-001/Test/Test_GT/{stem}/SoftMap/{stem}_ODsegSoftmap.png"
            cup = f"Test-20211018T060000Z-001/Test/Test_GT/{stem}/SoftMap/{stem}_cupsegSoftmap.png"
        if disc not in members or cup not in members:
            raise AssertionError((name, disc, cup))
        with Image.open(io.BytesIO(z.read(name))) as image:
            image.verify()
        rows.append({
            "dataset": "DRISHTI-GS1",
            "source": "lokeshsaipureddi Kaggle mirror",
            "image": name,
            "disc_mask": disc,
            "cup_mask": cup,
            "label": label,
            "official_style_split": split,
        })

assert len(rows) == 101
assert Counter(r["official_style_split"] for r in rows) == {"train": 50, "test": 51}
assert Counter(r["label"] for r in rows) == {"glaucoma": 70, "normal": 31}

manifest = audit / "drishti_gs1_kaggle_mirror_manifest.csv"
with manifest.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

chosen = [next(r for r in rows if r["label"] == label) for label in ("normal", "glaucoma")]
with zipfile.ZipFile(archive) as z:
    canvas = Image.new("RGB", (800, 430), "white")
    draw = ImageDraw.Draw(canvas)
    for i, row in enumerate(chosen):
        with Image.open(io.BytesIO(z.read(row["image"]))) as image:
            image = image.convert("RGB")
            image.thumbnail((370, 350))
            canvas.paste(image, (20 + i * 400 + (370 - image.width) // 2, 35))
        draw.text((20 + i * 400, 395), f"{row['label']} | {Path(row['image']).name}", fill="black")
    canvas.save(audit / "drishti_gs1_kaggle_mirror_samples.jpg", quality=92)

summary = {
    "dataset": "DRISHTI-GS1",
    "source": "Kaggle mirror lokeshsaipureddi",
    "images": len(rows),
    "labels": dict(Counter(r["label"] for r in rows)),
    "official_style_split": dict(Counter(r["official_style_split"] for r in rows)),
    "disc_masks": len(rows),
    "cup_masks": len(rows),
    "manifest": str(manifest),
    "provisional": True,
}
(audit / "drishti_gs1_kaggle_mirror_audit.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))


In [ ]:
%%writefile tools/audit_split_duplicates.py
"""Check exact archive-image duplicates across measured train/val/test splits."""
import csv
import hashlib
import json
import zipfile
from collections import defaultdict
from pathlib import Path


experiments = (
    ("ACRIMA", Path("data/raw/ACRIMA_Database.zip"), Path("outputs/acrima_resnet18_frozen/split.csv"), "member"),
    ("RIM-ONE r3", Path("data/raw/RIM-ONE-r3.zip"), Path("outputs/rim_unet_baseline/split.csv"), "image"),
)
report = {}
for dataset, archive_path, split_path, member_key in experiments:
    rows = list(csv.DictReader(split_path.open(newline="", encoding="utf-8")))
    groups = defaultdict(list)
    with zipfile.ZipFile(archive_path) as archive:
        for row in rows:
            digest = hashlib.sha256(archive.read(row[member_key])).hexdigest()
            groups[digest].append({"member": row[member_key], "split": row["split"]})
    duplicates = [members for members in groups.values() if len(members) > 1]
    cross_split = [members for members in duplicates if len({m["split"] for m in members}) > 1]
    report[dataset] = {"n_images": len(rows), "exact_duplicate_groups": duplicates, "cross_split_duplicate_groups": cross_split}
    print(f"{dataset}: {len(rows)} images; {len(duplicates)} exact duplicate groups; {len(cross_split)} cross-split groups")
    assert not cross_split, f"Exact-image leakage in {dataset}"

destination = Path("outputs/split_duplicate_audit.json")
destination.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Wrote", destination)


In [ ]:
# Build every manifest directly from the downloaded archives.
if os.name != 'nt' and shutil.which('bsdtar') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'libarchive-tools'], check=True)
for name in ('audit_acrima.py', 'audit_rim_one_r3.py', 'audit_origa_refuge.py', 'audit_refuge_original.py', 'audit_drishti_mirror.py'):
    run_script(name)
for filename in ('acrima_manifest.csv', 'rim_one_r3_manifest.csv', 'origa_manifest.csv', 'refuge_manifest.csv', 'drishti_gs1_kaggle_mirror_manifest.csv', 'refuge_test_labels_official.csv'):
    assert (ROOT / 'data/audit' / filename).is_file(), filename
print('Step 2 complete: five manifests and official REFUGE test labels are verified.')

### Visual dataset check

These sheets let you inspect image quality and annotation alignment before training. For ORIGA and REFUGE the mask values are `0=background, 1=disc rim, 2=cup`; therefore the **disc is `(mask > 0)`**, and the **cup is `(mask == 2)`**. ACRIMA has no masks, so it must not receive Dice or IoU scores.

In [ ]:
for name in ('acrima_samples.jpg', 'rim_sample.jpg', 'origa_sample.png', 'refuge_sample.png', 'drishti_gs1_kaggle_mirror_samples.jpg'):
    path = ROOT / 'data/audit' / name
    if path.exists():
        print(name)
        display(Image.open(path))

## Step 3 — Baseline classification and RIM-ONE segmentation

Train the small, reproducible baselines first. These produce frozen ResNet-18 features, a compact RIM-ONE U-Net, held-out metrics, checkpoints, and an exact duplicate audit. They are *baselines*, not the final model claim.

Run the source cells below, then the stage runner. Every file they create comes from code **inside this notebook**.

In [ ]:
%%writefile models/__init__.py
# Package marker for notebook-generated models.


In [ ]:
%%writefile models/unet_small.py
"""Compact two-channel U-Net used by the RIM-ONE baseline."""
import torch
from torch import nn


def block(a,b):
    return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(),nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU())


class UNet(nn.Module):
    def __init__(self):
        super().__init__(); self.e1=block(3,16); self.e2=block(16,32); self.e3=block(32,64); self.mid=block(64,128)
        self.pool=nn.MaxPool2d(2); self.d3=block(128+64,64); self.d2=block(64+32,32); self.d1=block(32+16,16); self.head=nn.Conv2d(16,2,1)
    def forward(self,x):
        a=self.e1(x); b=self.e2(self.pool(a)); c=self.e3(self.pool(b)); x=self.mid(self.pool(c))
        x=self.d3(torch.cat((nn.functional.interpolate(x,size=c.shape[-2:]),c),1))
        x=self.d2(torch.cat((nn.functional.interpolate(x,size=b.shape[-2:]),b),1))
        return self.head(self.d1(torch.cat((nn.functional.interpolate(x,size=a.shape[-2:]),a),1)))


In [ ]:
%%writefile tools/train_acrima_baseline.py
"""Reproducible ACRIMA-only frozen ResNet-18 feature baseline (CPU compatible)."""
from __future__ import annotations

import csv
import io
import json
import random
import zipfile
from pathlib import Path

import numpy as np
import joblib
import torch
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from torchvision.models import ResNet18_Weights, resnet18

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(4)
manifest = list(csv.DictReader(Path("data/audit/acrima_manifest.csv").open(encoding="utf-8")))
labels = np.array([int(r["label"] == "glaucoma") for r in manifest])
indices = np.arange(len(manifest))
trainval, test = train_test_split(indices, test_size=0.20, stratify=labels, random_state=SEED)
train, val = train_test_split(trainval, test_size=0.1875, stratify=labels[trainval], random_state=SEED)
out = Path("outputs/acrima_resnet18_frozen")
out.mkdir(parents=True, exist_ok=True)
split = {int(i): name for name, arr in (("train", train), ("val", val), ("test", test)) for i in arr}
with (out / "split.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["image_id", "member", "label", "split"])
    writer.writerows((r["image_id"], r["member"], r["label"], split[i]) for i, r in enumerate(manifest))

weights = ResNet18_Weights.IMAGENET1K_V1
model = resnet18(weights=weights)
model.fc = torch.nn.Identity()
model.eval()
transform = weights.transforms()
features = []
with zipfile.ZipFile("data/raw/ACRIMA_Database.zip") as archive, torch.no_grad():
    for start in range(0, len(manifest), 16):
        batch = []
        for row in manifest[start:start + 16]:
            with Image.open(io.BytesIO(archive.read(row["member"]))) as image:
                batch.append(transform(image.convert("RGB")))
        features.append(model(torch.stack(batch)).numpy())
        print(f"features {min(start+16,len(manifest))}/{len(manifest)}", flush=True)
X = np.concatenate(features)
candidate_cs = [0.01, 0.1, 1.0, 10.0]
best = None
for c in candidate_cs:
    clf = LogisticRegression(C=c, class_weight="balanced", max_iter=1000, random_state=SEED).fit(X[train], labels[train])
    auc = roc_auc_score(labels[val], clf.predict_proba(X[val])[:, 1])
    if best is None or auc > best[0]:
        best = (auc, c)
clf = LogisticRegression(C=best[1], class_weight="balanced", max_iter=1000, random_state=SEED).fit(X[trainval], labels[trainval])
joblib.dump(clf, out / "classifier.joblib")
p = clf.predict_proba(X[test])[:, 1]
yhat = (p >= 0.5).astype(int)
precision, recall, f1, _ = precision_recall_fscore_support(labels[test], yhat, average="binary", zero_division=0)
tn, fp, fn, tp = confusion_matrix(labels[test], yhat, labels=[0, 1]).ravel()
metrics = {"dataset": "ACRIMA", "experiment": "frozen ResNet-18 features + logistic regression", "seed": SEED,
           "n_train": len(train), "n_val": len(val), "n_test": len(test), "selection_val_auc": best[0], "selected_C": best[1],
           "test_accuracy": accuracy_score(labels[test], yhat), "test_precision": precision, "test_recall": recall,
           "test_f1": f1, "test_auc": roc_auc_score(labels[test], p), "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}
(out / "metrics.json").write_text(json.dumps(metrics, indent=2))
with (out / "test_predictions.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["image_id", "true_label", "probability_glaucoma", "predicted_label"])
    writer.writerows((manifest[i]["image_id"], labels[i], float(prob), int(pred)) for i, prob, pred in zip(test, p, yhat))
print(json.dumps(metrics, indent=2), flush=True)


In [ ]:
%%writefile tools/package_acrima_classifier.py
"""Package the measured frozen-feature classifier as one PyTorch checkpoint."""
import csv
import io
import json
import zipfile
from pathlib import Path

import joblib
import numpy as np
import torch
from PIL import Image
from torchvision.models import ResNet18_Weights, resnet18


torch.set_num_threads(4)
root = Path("outputs/acrima_resnet18_frozen")
clf = joblib.load(root / "classifier.joblib")
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = torch.nn.Linear(512, 1)
with torch.no_grad():
    model.fc.weight.copy_(torch.from_numpy(clf.coef_).float())
    model.fc.bias.copy_(torch.from_numpy(clf.intercept_).float())
model.eval()
transform = ResNet18_Weights.IMAGENET1K_V1.transforms()
manifest = {r["image_id"]: r for r in csv.DictReader(Path("data/audit/acrima_manifest.csv").open(encoding="utf-8"))}
predictions = list(csv.DictReader((root / "test_predictions.csv").open(encoding="utf-8")))

# Float32 conversion changes a logistic probability only by numerical rounding.
with zipfile.ZipFile("data/raw/ACRIMA_Database.zip") as archive, torch.no_grad():
    for record in predictions[:5]:
        image = Image.open(io.BytesIO(archive.read(manifest[record["image_id"]]["member"]))).convert("RGB")
        probability = float(model(transform(image)[None]).sigmoid().item())
        expected = float(record["probability_glaucoma"])
        assert abs(probability - expected) < 1e-5, (record["image_id"], probability, expected)

checkpoint = root / "resnet18_classifier.pt"
torch.save(model.state_dict(), checkpoint)
metadata = {
    "architecture": "torchvision.models.resnet18 with one-logit linear head",
    "preprocessing": "ResNet18_Weights.IMAGENET1K_V1.transforms()",
    "positive_class": "glaucoma",
    "probability": "sigmoid(logit)",
    "threshold": 0.5,
    "source_experiment": "outputs/acrima_resnet18_frozen/metrics.json",
    "checkpoint_bytes": checkpoint.stat().st_size,
    "verification": "First five saved ACRIMA test probabilities agree within 1e-5",
}
(root / "checkpoint_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(checkpoint, checkpoint.stat().st_size, "bytes; five probabilities verified")


In [ ]:
%%writefile tools/train_rim_segmentation.py
"""Train and evaluate a compact disc/cup U-Net on original RIM-ONE r3."""
import csv
import io
import json
import random
import sys
import zipfile
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from models.unet_small import UNet

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(4)
SIZE = 256
rows = list(csv.DictReader(Path('data/audit/rim_one_r3_manifest.csv').open(encoding='utf-8')))
labels = [r['label_original'] for r in rows]
idx = np.arange(len(rows))
trainval, test = train_test_split(idx, test_size=.2, stratify=labels, random_state=SEED)
train, val = train_test_split(trainval, test_size=.1875, stratify=np.array(labels)[trainval], random_state=SEED)
out = Path('outputs/rim_unet_baseline'); out.mkdir(parents=True, exist_ok=True)
with (out/'split.csv').open('w',newline='',encoding='utf-8') as f:
    w=csv.writer(f); w.writerow(['image','split']); w.writerows((rows[i]['image'],s) for s,a in [('train',train),('val',val),('test',test)] for i in a)

class RIM(Dataset):
    def __init__(self, indices, augment=False):
        self.indices=list(indices); self.augment=augment
        with zipfile.ZipFile('data/raw/RIM-ONE-r3.zip') as z:
            self.data=[]
            for i in self.indices:
                r=rows[i]
                im=Image.open(io.BytesIO(z.read(r['image']))).convert('RGB')
                im=im.crop((0,0,im.width//2,im.height)).resize((SIZE,SIZE))
                masks=[]
                for key in ('disc_mask','cup_mask'):
                    mask=Image.open(io.BytesIO(z.read(r[key]))).convert('L')
                    mask=mask.crop((0,0,mask.width//2,mask.height)).resize((SIZE,SIZE),Image.Resampling.NEAREST)
                    masks.append(np.asarray(mask)>127)
                self.data.append((np.asarray(im).copy(),np.stack(masks).astype('float32')))
    def __len__(self): return len(self.data)
    def __getitem__(self,i):
        im,mask=self.data[i]; im=im.copy(); mask=mask.copy()
        if self.augment and random.random()<.5: im=im[:,::-1].copy(); mask=mask[:,:,::-1].copy()
        return torch.from_numpy(np.ascontiguousarray(im.transpose(2,0,1))).float()/255,torch.from_numpy(mask)

def scores(model,ds):
    model.eval(); totals=np.zeros((2,2)); count=0
    with torch.no_grad():
        for x,y in DataLoader(ds,batch_size=4):
            p=(model(x).sigmoid()>.5).numpy(); g=y.numpy()>0
            for k in range(2):
                a=p[:,k]; b=g[:,k]; inter=(a&b).sum(axis=(1,2)); ps=a.sum(axis=(1,2)); gs=b.sum(axis=(1,2)); union=(a|b).sum(axis=(1,2))
                totals[k,0]+=((2*inter+1e-6)/(ps+gs+1e-6)).sum(); totals[k,1]+=((inter+1e-6)/(union+1e-6)).sum()
            count+=len(x)
    return totals/count

train_ds=RIM(train,True); val_ds=RIM(val); test_ds=RIM(test)
model=UNet(); opt=torch.optim.AdamW(model.parameters(),lr=1e-3); bce=nn.BCEWithLogitsLoss()
best=-1; history=[]
for epoch in range(12):
    model.train(); losses=[]
    for x,y in DataLoader(train_ds,batch_size=4,shuffle=True):
        logits=model(x); prob=logits.sigmoid(); dice=1-((2*(prob*y).sum((0,2,3))+1)/((prob+y).sum((0,2,3))+1)).mean(); loss=bce(logits,y)+dice
        opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
    v=scores(model,val_ds); rank=v[:,0].mean(); history.append({'epoch':epoch+1,'train_loss':float(np.mean(losses)),'val_disc_dice':float(v[0,0]),'val_cup_dice':float(v[1,0])})
    if rank>best: best=rank; torch.save(model.state_dict(),out/'best.pt')
    print(history[-1],flush=True)
model.load_state_dict(torch.load(out/'best.pt',weights_only=True)); result=scores(model,test_ds)
metrics={'dataset':'RIM-ONE r3','split_seed':SEED,'n_train':len(train),'n_val':len(val),'n_test':len(test),'test_disc_dice':float(result[0,0]),'test_disc_iou':float(result[0,1]),'test_cup_dice':float(result[1,0]),'test_cup_iou':float(result[1,1])}
(out/'metrics.json').write_text(json.dumps(metrics,indent=2)); (out/'history.json').write_text(json.dumps(history,indent=2)); print(json.dumps(metrics,indent=2),flush=True)


In [ ]:
%%writefile tools/evaluate_rim_segmentation.py
"""Save held-out per-image optic disc/cup metrics from the saved RIM model."""
import csv
import io
import json
import sys
import zipfile
from pathlib import Path

import numpy as np
import torch
from PIL import Image

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from models.unet_small import UNet


torch.set_num_threads(4)
root = Path("outputs/rim_unet_baseline")
manifest = {r["image"]: r for r in csv.DictReader(Path("data/audit/rim_one_r3_manifest.csv").open(encoding="utf-8"))}
test_names = [r["image"] for r in csv.DictReader((root / "split.csv").open(encoding="utf-8")) if r["split"] == "test"]
model = UNet()
model.load_state_dict(torch.load(root / "best.pt", map_location="cpu", weights_only=True))
model.eval()


def load_left(z, name, is_mask=False):
    image = Image.open(io.BytesIO(z.read(name))).convert("L" if is_mask else "RGB")
    return image.crop((0, 0, image.width // 2, image.height)).resize(
        (256, 256), Image.Resampling.NEAREST if is_mask else Image.Resampling.BICUBIC
    )


rows = []
with zipfile.ZipFile("data/raw/RIM-ONE-r3.zip") as archive, torch.no_grad():
    for name in test_names:
        record = manifest[name]
        image = np.asarray(load_left(archive, name)).copy()
        truth = [np.asarray(load_left(archive, record[key], True)) > 127 for key in ("disc_mask", "cup_mask")]
        x = torch.from_numpy(image.transpose(2, 0, 1).copy()).float()[None] / 255
        pred = model(x).sigmoid()[0].numpy() > 0.5
        result = {"image": name, "label_original": record["label_original"]}
        for i, region in enumerate(("disc", "cup")):
            p, y = pred[i], truth[i]
            intersection = int(np.logical_and(p, y).sum())
            predicted = int(p.sum())
            actual = int(y.sum())
            union = int(np.logical_or(p, y).sum())
            result[f"{region}_dice"] = (2 * intersection + 1e-6) / (predicted + actual + 1e-6)
            result[f"{region}_iou"] = (intersection + 1e-6) / (union + 1e-6)
        rows.append(result)

saved = json.loads((root / "metrics.json").read_text(encoding="utf-8"))
assert len(rows) == saved["n_test"]
for region in ("disc", "cup"):
    for metric in ("dice", "iou"):
        key = f"{region}_{metric}"
        assert abs(float(np.mean([r[key] for r in rows])) - saved[f"test_{key}"]) < 1e-6, key

destination = root / "test_predictions.csv"
with destination.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
print(f"Verified and wrote {len(rows)} held-out per-image scores to {destination}")


In [ ]:
%%writefile tools/evaluate_acrima_on_rim.py
"""External test: ACRIMA-trained baseline on RIM-ONE r3 normal/glaucoma eyes."""
import csv
import io
import json
import zipfile
from pathlib import Path

import joblib
import numpy as np
import torch
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support, roc_auc_score
from torchvision.models import ResNet18_Weights, resnet18

torch.set_num_threads(4)
rows = [r for r in csv.DictReader(Path("data/audit/rim_one_r3_manifest.csv").open(encoding="utf-8")) if r["label_original"] != "suspect"]
labels = np.array([int(r["label_original"] == "glaucoma") for r in rows])
weights = ResNet18_Weights.IMAGENET1K_V1
model = resnet18(weights=weights)
model.fc = torch.nn.Identity()
model.eval()
transform = weights.transforms()
feats = []
with zipfile.ZipFile("data/raw/RIM-ONE-r3.zip") as archive, torch.no_grad():
    for start in range(0, len(rows), 16):
        batch = []
        for row in rows[start:start+16]:
            with Image.open(io.BytesIO(archive.read(row["image"]))) as image:
                image = image.convert("RGB")
                image = image.crop((0, 0, image.width // 2, image.height))
                batch.append(transform(image))
        feats.append(model(torch.stack(batch)).numpy())
        print(f"features {min(start+16,len(rows))}/{len(rows)}", flush=True)
classifier = joblib.load("outputs/acrima_resnet18_frozen/classifier.joblib")
probs = classifier.predict_proba(np.concatenate(feats))[:, 1]
preds = (probs >= 0.5).astype(int)
precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
metrics = {"train_dataset": "ACRIMA", "test_dataset": "RIM-ONE r3", "protocol": "left stereo view; suspects excluded; no RIM training or threshold tuning",
           "n_test": len(rows), "accuracy": accuracy_score(labels, preds), "precision": precision, "recall": recall,
           "f1": f1, "auc": roc_auc_score(labels, probs), "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}
out = Path("outputs/acrima_resnet18_frozen")
(out / "rim_external_metrics.json").write_text(json.dumps(metrics, indent=2))
with (out / "rim_external_predictions.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["image", "true_label", "probability_glaucoma", "predicted_label"])
    writer.writerows((row["image"], int(y), float(p), int(pred)) for row, y, p, pred in zip(rows, labels, probs, preds))
print(json.dumps(metrics, indent=2))


In [ ]:
%%writefile tools/evaluate_acrima_on_repackaged.py
"""Untuned external classification on ORIGA and labeled REFUGE validation images."""
import csv
import io
import json
import zipfile
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support, roc_auc_score
from torchvision.models import ResNet18_Weights, resnet18


torch.set_num_threads(4)
base = Path("outputs/acrima_resnet18_frozen")
archive_path = Path("data/raw/ORIGA_candidate_figshare.zip")
model = resnet18(weights=None)
model.fc = torch.nn.Linear(512, 1)
model.load_state_dict(torch.load(base / "resnet18_classifier.pt", map_location="cpu", weights_only=True))
model.eval()
transform = ResNet18_Weights.IMAGENET1K_V1.transforms()


def evaluate(name, rows, output_stem):
    result_rows = []
    with zipfile.ZipFile(archive_path) as archive, torch.no_grad():
        for start in range(0, len(rows), 16):
            batch = rows[start:start + 16]
            images = []
            for record in batch:
                with Image.open(io.BytesIO(archive.read(record["image"]))) as source:
                    images.append(transform(source.convert("RGB")))
            probabilities = model(torch.stack(images)).sigmoid().flatten().numpy()
            for record, probability in zip(batch, probabilities):
                result_rows.append({
                    "image": record["image"], "true_label": int(record["label"]),
                    "probability_glaucoma": float(probability),
                    "predicted_label": int(probability >= 0.5),
                })
            print(f"{name}: {min(start + len(batch), len(rows))}/{len(rows)}", flush=True)
    y = np.array([r["true_label"] for r in result_rows])
    p = np.array([r["probability_glaucoma"] for r in result_rows])
    predicted = np.array([r["predicted_label"] for r in result_rows])
    precision, recall, f1, _ = precision_recall_fscore_support(y, predicted, average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, predicted, labels=[0, 1]).ravel()
    metrics = {
        "train_dataset": "ACRIMA", "test_dataset": name,
        "protocol": "frozen ACRIMA checkpoint; ImageNet ResNet-18 center-crop preprocessing; no target training or threshold tuning",
        "n_test": len(rows), "accuracy": float(accuracy_score(y, predicted)),
        "precision": float(precision), "recall": float(recall), "f1": float(f1),
        "auc": float(roc_auc_score(y, p)),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }
    (base / f"{output_stem}_external_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    with (base / f"{output_stem}_external_predictions.csv").open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(result_rows[0]))
        writer.writeheader()
        writer.writerows(result_rows)
    print(json.dumps(metrics, indent=2), flush=True)


origa = list(csv.DictReader(Path("data/audit/origa_manifest.csv").open(newline="", encoding="utf-8")))
refuge = [r for r in csv.DictReader(Path("data/audit/refuge_manifest.csv").open(newline="", encoding="utf-8")) if r["split"] == "val"]
assert len(origa) == 650 and len(refuge) == 400
evaluate("ORIGA", origa, "origa")
evaluate("REFUGE validation", refuge, "refuge_val")


In [ ]:
if RUN_BASELINES:
    run_script('train_acrima_baseline.py', 'outputs/acrima_resnet18_frozen/metrics.json')
    run_script('package_acrima_classifier.py', 'outputs/acrima_resnet18_frozen/resnet18_classifier.pt')
    run_script('train_rim_segmentation.py', 'outputs/rim_unet_baseline/metrics.json')
    run_script('evaluate_rim_segmentation.py', 'outputs/rim_unet_baseline/test_predictions.csv')
    run_script('evaluate_acrima_on_rim.py', 'outputs/acrima_resnet18_frozen/rim_external_metrics.json')
    run_script('evaluate_acrima_on_repackaged.py', 'outputs/acrima_resnet18_frozen/refuge_val_external_metrics.json')
    run_script('audit_split_duplicates.py')
print('Step 3 complete: baseline metrics and checkpoints are saved.')

## Step 4 — Four-dataset pooled classification comparison

A pooled frozen-feature classifier uses full-image letterboxing and balances dataset/class groups. ORIGA B and REFUGE validation remain held out. REFUGE official test labels provide an independent final check. The same-image chart prevents unfair comparisons between different test sets.

Run the source cells below, then the stage runner. Every file they create comes from code **inside this notebook**.

In [ ]:
%%writefile tools/train_pooled_frozen_baseline.py
"""Four-dataset pooled frozen ResNet-18 baseline with declared held-out splits."""
import csv
import hashlib
import io
import json
import random
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import joblib
import numpy as np
import torch
from PIL import Image, ImageOps
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.transforms import Normalize, ToTensor


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(4)
root = Path("outputs/pooled_four_dataset_frozen")
root.mkdir(parents=True, exist_ok=True)
archive_paths = {
    "ACRIMA": Path("data/raw/ACRIMA_Database.zip"),
    "RIM-ONE r3": Path("data/raw/RIM-ONE-r3.zip"),
    "ORIGA": Path("data/raw/ORIGA_candidate_figshare.zip"),
    "REFUGE": Path("data/raw/ORIGA_candidate_figshare.zip"),
}


def add_record(records, dataset, member, label, split, left_stereo=False):
    records.append({"dataset": dataset, "member": member, "label": int(label), "split": split, "left_stereo": left_stereo})


records = []
for row in csv.DictReader(Path("outputs/acrima_resnet18_frozen/split.csv").open(encoding="utf-8")):
    add_record(records, "ACRIMA", row["member"], row["label"] == "glaucoma", row["split"])

origa = list(csv.DictReader(Path("data/audit/origa_manifest.csv").open(encoding="utf-8")))
origa_a = [r for r in origa if r["provided_set"] == "A"]
origa_b = [r for r in origa if r["provided_set"] == "B"]
assert len(origa_a) == len(origa_b) == 325
a_train, a_val = train_test_split(np.arange(len(origa_a)), test_size=0.2, stratify=[r["label"] for r in origa_a], random_state=SEED)
for i, row in enumerate(origa_a):
    add_record(records, "ORIGA", row["image"], row["label"], "val" if i in set(a_val) else "train")
for row in origa_b:
    add_record(records, "ORIGA", row["image"], row["label"], "test")

refuge = list(csv.DictReader(Path("data/audit/refuge_manifest.csv").open(encoding="utf-8")))
refuge_train = [r for r in refuge if r["split"] == "train"]
refuge_val = [r for r in refuge if r["split"] == "val"]
assert len(refuge_train) == len(refuge_val) == 400
r_train, r_val = train_test_split(np.arange(len(refuge_train)), test_size=0.2, stratify=[r["label"] for r in refuge_train], random_state=SEED)
for i, row in enumerate(refuge_train):
    add_record(records, "REFUGE", row["image"], row["label"], "val" if i in set(r_val) else "train")
for row in refuge_val:
    add_record(records, "REFUGE", row["image"], row["label"], "test")

rim_labels = {r["image"]: r["label_original"] for r in csv.DictReader(Path("data/audit/rim_one_r3_manifest.csv").open(encoding="utf-8"))}
for row in csv.DictReader(Path("outputs/rim_unet_baseline/split.csv").open(encoding="utf-8")):
    label = rim_labels[row["image"]]
    if label in ("normal", "glaucoma"):
        add_record(records, "RIM-ONE r3", row["image"], label == "glaucoma", row["split"], True)

assert len(records) == 705 + 650 + 800 + 124
assert len({(r["dataset"], r["member"]) for r in records}) == len(records)
with (root / "split.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(records[0]))
    writer.writeheader()
    writer.writerows(records)

model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = torch.nn.Identity()
model.eval()
to_tensor = ToTensor()
normalize = Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
features = []
digests = defaultdict(list)
with zipfile.ZipFile(archive_paths["ACRIMA"]) as acrima_zip, zipfile.ZipFile(archive_paths["RIM-ONE r3"]) as rim_zip, zipfile.ZipFile(archive_paths["ORIGA"]) as combined_zip, torch.no_grad():
    archives = {"ACRIMA": acrima_zip, "RIM-ONE r3": rim_zip, "ORIGA": combined_zip, "REFUGE": combined_zip}
    for start in range(0, len(records), 16):
        batch = []
        for row in records[start:start + 16]:
            raw = archives[row["dataset"]].read(row["member"])
            digests[hashlib.sha256(raw).hexdigest()].append((row["dataset"], row["member"], row["split"]))
            with Image.open(io.BytesIO(raw)) as source:
                image = source.convert("RGB")
                if row["left_stereo"]:
                    image = image.crop((0, 0, image.width // 2, image.height))
                image = ImageOps.pad(image, (224, 224), method=Image.Resampling.BICUBIC, color=(0, 0, 0))
                batch.append(normalize(to_tensor(image)))
        features.append(model(torch.stack(batch)).numpy())
        if (start // 16) % 10 == 0:
            print(f"features {min(start + 16, len(records))}/{len(records)}", flush=True)
X = np.concatenate(features)
duplicates = [group for group in digests.values() if len(group) > 1]
cross_split = [group for group in duplicates if len({item[2] for item in group}) > 1]
assert not cross_split, f"Cross-split exact duplicates: {cross_split[:3]}"
(root / "duplicate_audit.json").write_text(json.dumps({"exact_duplicate_groups": duplicates, "cross_split_groups": cross_split}, indent=2), encoding="utf-8")

dataset = np.array([r["dataset"] for r in records])
labels = np.array([r["label"] for r in records])
splits = np.array([r["split"] for r in records])
train = np.where(splits == "train")[0]
val = np.where(splits == "val")[0]
trainval = np.where(splits != "test")[0]


def balancing_weights(indices):
    counts = Counter((dataset[i], int(labels[i])) for i in indices)
    weights = np.array([1.0 / counts[(dataset[i], int(labels[i]))] for i in indices])
    return weights * (len(weights) / weights.sum())


trials = []
for c in (0.01, 0.1, 1.0, 10.0):
    classifier = LogisticRegression(C=c, max_iter=1000, random_state=SEED)
    classifier.fit(X[train], labels[train], sample_weight=balancing_weights(train))
    probability = classifier.predict_proba(X[val])[:, 1]
    auc_by_dataset = {name: float(roc_auc_score(labels[val][dataset[val] == name], probability[dataset[val] == name])) for name in sorted(set(dataset[val]))}
    macro_auc = float(np.mean(list(auc_by_dataset.values())))
    trials.append({"C": c, "macro_val_auc": macro_auc, "per_dataset_val_auc": auc_by_dataset})
best = max(trials, key=lambda r: r["macro_val_auc"])
classifier = LogisticRegression(C=best["C"], max_iter=1000, random_state=SEED)
classifier.fit(X[trainval], labels[trainval], sample_weight=balancing_weights(trainval))
joblib.dump(classifier, root / "classifier.joblib")

full_model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
full_model.fc = torch.nn.Linear(512, 1)
with torch.no_grad():
    full_model.fc.weight.copy_(torch.from_numpy(classifier.coef_).float())
    full_model.fc.bias.copy_(torch.from_numpy(classifier.intercept_).float())
torch.save(full_model.state_dict(), root / "resnet18_classifier.pt")

metrics = {}
for name in sorted(set(dataset)):
    indices = np.where((dataset == name) & (splits == "test"))[0]
    y = labels[indices]
    probabilities = classifier.predict_proba(X[indices])[:, 1]
    predicted = (probabilities >= 0.5).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(y, predicted, average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, predicted, labels=[0, 1]).ravel()
    metrics[name] = {"n_test": len(indices), "accuracy": float(accuracy_score(y, predicted)), "precision": float(precision), "recall": float(recall), "f1": float(f1), "auc": float(roc_auc_score(y, probabilities)), "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}
    with (root / (name.lower().replace(" ", "_").replace("-", "_") + "_test_predictions.csv")).open("w", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        writer.writerow(["dataset", "member", "true_label", "probability_glaucoma", "predicted_label"])
        writer.writerows((name, records[i]["member"], int(labels[i]), float(p), int(yhat)) for i, p, yhat in zip(indices, probabilities, predicted))

summary = {
    "status": "four_dataset_partial_baseline",
    "architecture": "frozen ImageNet ResNet-18 features + balanced logistic head",
    "preprocessing": "full RGB image letterboxed to 224x224; RIM-ONE uses left stereo view",
    "protocol": "ACRIMA fixed split; ORIGA A train/val B test; REFUGE train train/val and official validation held out; RIM-ONE fixed split excluding suspects; seed 42",
    "selection": {"metric": "macro validation AUC across four datasets", "trials": trials, "selected_C": best["C"]},
    "split_counts": {name: dict(Counter(splits[dataset == name])) for name in sorted(set(dataset))},
    "test_metrics": metrics,
    "limitations": "DRISHTI-GS unavailable; ORIGA/REFUGE come from a verified third-party repackaging; ORIGA patient identifiers unavailable; frozen backbone is not final fine-tuning",
}
(root / "metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2), flush=True)


In [ ]:
%%writefile tools/evaluate_pooled_refuge_official_test.py
"""Evaluate the frozen four-dataset model on original-source REFUGE test labels."""
import csv
import io
import json
import zipfile
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support, roc_auc_score
from torchvision.models import resnet18
from torchvision.transforms import Normalize, ToTensor


torch.set_num_threads(4)
root = Path("outputs/pooled_four_dataset_frozen")
records = [r for r in csv.DictReader(Path("data/audit/refuge_manifest.csv").open(newline="", encoding="utf-8")) if r["split"] == "test"]
assert len(records) == 400 and all(r["label"] in ("0", "1") for r in records)
model = resnet18(weights=None)
model.fc = torch.nn.Linear(512, 1)
model.load_state_dict(torch.load(root / "resnet18_classifier.pt", map_location="cpu", weights_only=True))
model.eval()
to_tensor = ToTensor()
normalize = Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
outputs = []
with zipfile.ZipFile("data/raw/ORIGA_candidate_figshare.zip") as archive, torch.no_grad():
    for start in range(0, len(records), 16):
        batch = records[start:start + 16]
        images = []
        for record in batch:
            with Image.open(io.BytesIO(archive.read(record["image"]))) as source:
                image = ImageOps.pad(source.convert("RGB"), (224, 224), method=Image.Resampling.BICUBIC, color=(0, 0, 0))
                images.append(normalize(to_tensor(image)))
        probability = model(torch.stack(images)).sigmoid().flatten().numpy()
        for record, p in zip(batch, probability):
            outputs.append({"image": record["image"], "true_label": int(record["label"]), "probability_glaucoma": float(p), "predicted_label": int(p >= 0.5)})
        print(f"REFUGE official test: {min(start + len(batch), len(records))}/{len(records)}", flush=True)

y = np.array([r["true_label"] for r in outputs])
p = np.array([r["probability_glaucoma"] for r in outputs])
predicted = np.array([r["predicted_label"] for r in outputs])
precision, recall, f1, _ = precision_recall_fscore_support(y, predicted, average="binary", zero_division=0)
tn, fp, fn, tp = confusion_matrix(y, predicted, labels=[0, 1]).ravel()
metrics = {
    "train_protocol": "pooled four-dataset frozen ResNet-18 baseline; no REFUGE validation/test tuning",
    "test_dataset": "REFUGE official test 400",
    "label_source": "original-source REFUGE-Test-GT/Glaucoma_label_and_Fovea_location.xlsx",
    "n_test": len(outputs), "accuracy": float(accuracy_score(y, predicted)),
    "precision": float(precision), "recall": float(recall), "f1": float(f1),
    "auc": float(roc_auc_score(y, p)),
    "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
}
(root / "refuge_official_test_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
with (root / "refuge_official_test_predictions.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(outputs[0]))
    writer.writeheader()
    writer.writerows(outputs)
print(json.dumps(metrics, indent=2))


In [ ]:
%%writefile tools/compare_classifiers_common_splits.py
"""Compare two classifiers only on identical held-out images in four datasets."""
import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, precision_recall_fscore_support, roc_auc_score


def rows(path):
    return list(csv.DictReader(path.open(newline="", encoding="utf-8")))


original = Path("outputs/acrima_resnet18_frozen")
pooled = Path("outputs/pooled_four_dataset_frozen")
acrima_members = {r["image_id"]: r["member"] for r in rows(original / "split.csv")}
sources = {
    "ACRIMA": (original / "test_predictions.csv", pooled / "acrima_test_predictions.csv", lambda r: acrima_members[r["image_id"]]),
    "ORIGA": (original / "origa_external_predictions.csv", pooled / "origa_test_predictions.csv", lambda r: r["image"]),
    "REFUGE": (original / "refuge_val_external_predictions.csv", pooled / "refuge_test_predictions.csv", lambda r: r["image"]),
    "RIM-ONE r3": (original / "rim_external_predictions.csv", pooled / "rim_one_r3_test_predictions.csv", lambda r: r["image"]),
}


def calculate(records):
    y = np.array([int(r["true_label"]) for r in records])
    p = np.array([float(r["probability_glaucoma"]) for r in records])
    prediction = np.array([int(r["predicted_label"]) for r in records])
    precision, recall, f1, _ = precision_recall_fscore_support(y, prediction, average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    return {"n_test": len(records), "accuracy": float(accuracy_score(y, prediction)), "balanced_accuracy": float(balanced_accuracy_score(y, prediction)), "precision": float(precision), "recall": float(recall), "f1": float(f1), "auc": float(roc_auc_score(y, p)), "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}


comparison = {}
for dataset, (original_path, pooled_path, key) in sources.items():
    baseline = {key(r): r for r in rows(original_path)}
    pooled_rows = rows(pooled_path)
    assert len(pooled_rows) == len({r["member"] for r in pooled_rows})
    original_rows = [baseline[r["member"]] for r in pooled_rows]
    assert all(int(a["true_label"]) == int(b["true_label"]) for a, b in zip(original_rows, pooled_rows)), dataset
    comparison[dataset] = {"ACRIMA-only frozen": calculate(original_rows), "Pooled four-dataset frozen": calculate(pooled_rows)}

destination = Path("outputs/classifier_common_split_comparison.json")
destination.write_text(json.dumps({"protocol": "Both models evaluated on identical held-out images; ORIGA B, REFUGE official validation, ACRIMA fixed test, RIM-ONE non-suspect fixed test subset", "results": comparison}, indent=2), encoding="utf-8")

datasets = list(comparison)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
x = np.arange(len(datasets))
for ax, metric, title in zip(axes, ("auc", "balanced_accuracy"), ("ROC AUC", "Balanced accuracy")):
    for offset, model, color in ((-0.18, "ACRIMA-only frozen", "#b66a36"), (0.18, "Pooled four-dataset frozen", "#14858a")):
        values = [comparison[d][model][metric] for d in datasets]
        ax.bar(x + offset, values, width=0.35, label=model, color=color)
    ax.set_xticks(x, datasets, rotation=15)
    ax.set_ylim(0, 1)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)
axes[1].legend(loc="upper left", fontsize=8)
fig.suptitle("Same held-out images per dataset; frozen-feature baselines", fontsize=13)
figure = Path("outputs/classifier_common_split_comparison.png")
fig.savefig(figure, dpi=170)
plt.close(fig)
print(destination)
print(figure)
for dataset, result in comparison.items():
    print(dataset, "n=", result["ACRIMA-only frozen"]["n_test"], "AUC", round(result["ACRIMA-only frozen"]["auc"], 3), "->", round(result["Pooled four-dataset frozen"]["auc"], 3))


In [ ]:
if RUN_BASELINES:
    run_script('train_pooled_frozen_baseline.py', 'outputs/pooled_four_dataset_frozen/metrics.json')
    run_script('evaluate_pooled_refuge_official_test.py', 'outputs/pooled_four_dataset_frozen/refuge_official_test_metrics.json')
    run_script('compare_classifiers_common_splits.py', 'outputs/classifier_common_split_comparison.json')
comparison = json.loads((ROOT / 'outputs/classifier_common_split_comparison.json').read_text())
for dataset, row in comparison['results'].items():
    print(dataset, 'AUC:', round(row['ACRIMA-only frozen']['auc'], 3), '->', round(row['Pooled four-dataset frozen']['auc'], 3))
display(Image.open(ROOT / 'outputs/classifier_common_split_comparison.png'))
print('Step 4 complete: fair held-out comparison saved.')

## Step 5 — Disc/cup segmentation on mask datasets

Train a pooled two-channel U-Net on RIM-ONE, ORIGA, and REFUGE. The notebook version uses CUDA when available. The first mask channel is the **whole optic disc** (rim plus cup); the second is cup. Evaluate on fixed held-out images and save per-image Dice/IoU and best/worst overlays. DRISHTI mirror segmentation is trained and reported separately.

Run the source cells below, then the stage runner. Every file they create comes from code **inside this notebook**.

In [ ]:
%%writefile tools/train_pooled_segmentation.py
"""Train a two-channel disc/cup U-Net across RIM-ONE, ORIGA, and REFUGE."""
from __future__ import annotations

import csv
import io
import json
import random
import sys
import zipfile
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from models.unet_small import UNet


SEED = 42
EPOCHS = 8
SIZE = 256
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

root = Path("outputs/pooled_segmentation")
root.mkdir(parents=True, exist_ok=True)
combined = zipfile.ZipFile("data/raw/ORIGA_candidate_figshare.zip")
rim = zipfile.ZipFile("data/raw/RIM-ONE-r3.zip")

records = []

rim_manifest = {
    row["image"]: row
    for row in csv.DictReader(Path("data/audit/rim_one_r3_manifest.csv").open(encoding="utf-8"))
}
rim_split = list(csv.DictReader(Path("outputs/rim_unet_baseline/split.csv").open(encoding="utf-8")))
for row in rim_split:
    records.append({
        "dataset": "RIM-ONE r3",
        "image": row["image"],
        "split": row["split"],
    })

origa = list(csv.DictReader(Path("data/audit/origa_manifest.csv").open(encoding="utf-8")))
origa_a = [row for row in origa if row["provided_set"] == "A"]
origa_b = [row for row in origa if row["provided_set"] == "B"]
train_a, val_a = train_test_split(
    np.arange(len(origa_a)),
    test_size=0.2,
    stratify=[row["label"] for row in origa_a],
    random_state=SEED,
)
for i, row in enumerate(origa_a):
    records.append({
        "dataset": "ORIGA",
        "image": row["image"],
        "mask": row["mask"],
        "split": "val" if i in set(val_a) else "train",
    })
for row in origa_b:
    records.append({
        "dataset": "ORIGA",
        "image": row["image"],
        "mask": row["mask"],
        "split": "test",
    })

refuge = list(csv.DictReader(Path("data/audit/refuge_manifest.csv").open(encoding="utf-8")))
train_rows = [row for row in refuge if row["split"] == "train"]
train_labels = [int(row["label"]) for row in train_rows]
train_idx, val_idx = train_split = train_test_split(
    np.arange(len(train_rows)),
    test_size=0.2,
    stratify=train_labels,
    random_state=SEED,
)
for i, row in enumerate(train_rows):
    records.append({
        "dataset": "REFUGE",
        "image": row["image"],
        "mask": row["mask"],
        "split": "val" if i in set(val_idx) else "train",
    })
for row in (r for r in refuge if r["split"] in ("val", "test")):
    records.append({
        "dataset": "REFUGE",
        "image": row["image"],
        "mask": row["mask"],
        "split": "test",
    })

assert len(records) == 159 + 650 + 1200
assert len({(row["dataset"], row["image"]) for row in records}) == len(records)
with (root / "split.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=["dataset", "image", "mask", "split"])
    writer.writeheader()
    writer.writerows({key: row.get(key, "") for key in ("dataset", "image", "mask", "split")} for row in records)


class PooledSegmentation(Dataset):
    def __init__(self, indices: list[int], augment: bool = False):
        self.indices = indices
        self.augment = augment
        self.data = []
        for i in indices:
            row = records[i]
            if row["dataset"] == "RIM-ONE r3":
                rim_row = rim_manifest[row["image"]]
                with Image.open(io.BytesIO(rim.read(row["image"]))).convert("RGB") as image:
                    image = image.crop((0, 0, image.width // 2, image.height)).resize(
                        (SIZE, SIZE), Image.Resampling.BICUBIC
                    )
                masks = []
                for key in ("disc_mask", "cup_mask"):
                    with Image.open(io.BytesIO(rim.read(rim_row[key]))).convert("L") as mask:
                        mask = mask.crop((0, 0, mask.width // 2, mask.height)).resize(
                            (SIZE, SIZE), Image.Resampling.NEAREST
                        )
                        masks.append(np.asarray(mask) > 127)
                truth = np.stack(masks).astype("float32")
            else:
                with Image.open(io.BytesIO(combined.read(row["image"]))).convert("RGB") as image:
                    image = image.resize((SIZE, SIZE), Image.Resampling.BICUBIC)
                with Image.open(io.BytesIO(combined.read(row["mask"]))).convert("L") as mask:
                    mask = mask.resize((SIZE, SIZE), Image.Resampling.NEAREST)
                    values = np.asarray(mask)
                    if row["dataset"] == "REFUGE" and set(np.unique(values)).issubset({0, 1, 2}):
                        pass
                truth = np.stack((values > 0, values == 2)).astype("float32")
            pixels = np.asarray(image).copy()
            self.data.append((pixels, truth))

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, i: int):
        pixels, truth = self.data[i]
        pixels = pixels.copy()
        truth = truth.copy()
        if self.augment and random.random() < 0.5:
            pixels = pixels[:, ::-1].copy()
            truth = truth[:, :, ::-1].copy()
        return (
            torch.from_numpy(np.ascontiguousarray(pixels.transpose(2, 0, 1))).float() / 255,
            torch.from_numpy(truth),
        )


train = [i for i, row in enumerate(records) if row["split"] == "train"]
val = [i for i, row in enumerate(records) if row["split"] == "val"]
test = [i for i, row in enumerate(records) if row["split"] == "test"]

train_ds = PooledSegmentation(train, augment=True)
val_ds = PooledSegmentation(val)
test_ds = PooledSegmentation(test)

model = UNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
bce = nn.BCEWithLogitsLoss()
history = []
best = -1.0


def scores(model: nn.Module, dataset: Dataset) -> np.ndarray:
    model.eval()
    totals = np.zeros((2, 2), dtype=np.float64)
    count = 0
    with torch.no_grad():
        for x, y in DataLoader(dataset, batch_size=4):
            pred = (model(x.to(device)).sigmoid() > 0.5).cpu().numpy()
            truth = y.numpy() > 0
            for channel in range(2):
                p = pred[:, channel]
                g = truth[:, channel]
                intersection = (p & g).sum(axis=(1, 2))
                predicted = p.sum(axis=(1, 2))
                actual = g.sum(axis=(1, 2))
                union = (p | g).sum(axis=(1, 2))
                totals[channel, 0] += ((2 * intersection + 1e-6) / (predicted + actual + 1e-6)).sum()
                totals[channel, 1] += ((intersection + 1e-6) / (union + 1e-6)).sum()
            count += len(x)
    return totals / count


for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    for x, y in DataLoader(train_ds, batch_size=4, shuffle=True):
        x, y = x.to(device), y.to(device)
        logits = model(x)
        prob = logits.sigmoid()
        dice = 1 - (
            (2 * (prob * y).sum((0, 2, 3)) + 1)
            / ((prob + y).sum((0, 2, 3)) + 1)
        ).mean()
        loss = bce(logits, y) + dice
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    val_scores = scores(model, val_ds)
    rank = float(val_scores[:, 0].mean())
    history.append({
        "epoch": epoch,
        "train_loss": float(np.mean(losses)),
        "val_disc_dice": float(val_scores[0, 0]),
        "val_cup_dice": float(val_scores[1, 0]),
    })
    if rank > best:
        best = rank
        torch.save(model.state_dict(), root / "best.pt")
    print(history[-1], flush=True)

model.load_state_dict(torch.load(root / "best.pt", map_location=device, weights_only=True))
with (root / "history.json").open("w", encoding="utf-8") as stream:
    json.dump(history, stream, indent=2)

test_rows = []
test_indices_by_dataset = {}
for dataset in ("RIM-ONE r3", "ORIGA", "REFUGE"):
    indices = [i for i in test if records[i]["dataset"] == dataset]
    test_indices_by_dataset[dataset] = indices
    subset = PooledSegmentation(indices)
    score = scores(model, subset)
    test_rows.append({
        "dataset": dataset,
        "n_test": len(indices),
        "test_disc_dice": float(score[0, 0]),
        "test_disc_iou": float(score[0, 1]),
        "test_cup_dice": float(score[1, 0]),
        "test_cup_iou": float(score[1, 1]),
    })

counts = {
    split: {
        "n": sum(row["split"] == split for row in records),
        "by_dataset": {
            dataset: sum(row["split"] == split and row["dataset"] == dataset for row in records)
            for dataset in ("RIM-ONE r3", "ORIGA", "REFUGE")
        },
    }
    for split in ("train", "val", "test")
}
summary = {
    "status": "three_dataset_partial_segmentation",
    "architecture": "compact randomly initialized UNet",
    "device": str(device),
    "size": SIZE,
    "seed": SEED,
    "epochs": EPOCHS,
    "datasets": ["RIM-ONE r3", "ORIGA", "REFUGE"],
    "protocol": "RIM-ONE existing 103/24/32; ORIGA A 260/65, B 325 test; REFUGE supplied train 320/80 reallocated, supplied validation and official test 800 test",
    "split_counts": counts,
    "test_metrics": test_rows,
    "limitations": "DRISHTI-GS unavailable; randomly initialized compact U-Net; REFUGE/ORIGA from the verified repackaging; no patient overlap audit for ORIGA.",
}
(root / "metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2), flush=True)

combined.close()
rim.close()


In [ ]:
%%writefile tools/render_pooled_segmentation.py
"""Render pooled segmentation panels and save per-image held-out scores."""
from __future__ import annotations

import csv
import io
import json
import sys
import zipfile
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageDraw

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from models.unet_small import UNet


SIZE = 256
root = Path("outputs/pooled_segmentation")
out = root / "examples"
out.mkdir(parents=True, exist_ok=True)
split = list(csv.DictReader((root / "split.csv").open(encoding="utf-8")))
rim_manifest = {
    row["image"]: row
    for row in csv.DictReader(Path("data/audit/rim_one_r3_manifest.csv").open(encoding="utf-8"))
}
model = UNet()
model.load_state_dict(torch.load(root / "best.pt", map_location="cpu", weights_only=True))
model.eval()


def load_record(row):
    if row["dataset"] == "RIM-ONE r3":
        archive = rim
        rim_row = rim_manifest[row["image"]]
        with Image.open(io.BytesIO(archive.read(row["image"]))).convert("RGB") as image:
            image = image.crop((0, 0, image.width // 2, image.height)).resize(
                (SIZE, SIZE), Image.Resampling.BICUBIC
            )
        masks = []
        for key in ("disc_mask", "cup_mask"):
            with Image.open(io.BytesIO(archive.read(rim_row[key]))).convert("L") as mask:
                mask = mask.crop((0, 0, mask.width // 2, mask.height)).resize(
                    (SIZE, SIZE), Image.Resampling.NEAREST
                )
                masks.append(np.asarray(mask) > 127)
        truth = np.stack(masks).astype("float32")
    else:
        with Image.open(io.BytesIO(combined.read(row["image"]))).convert("RGB") as image:
            image = image.resize((SIZE, SIZE), Image.Resampling.BICUBIC)
        with Image.open(io.BytesIO(combined.read(row["mask"]))).convert("L") as mask:
            mask = mask.resize((SIZE, SIZE), Image.Resampling.NEAREST)
            values = np.asarray(mask)
        truth = np.stack((values > 0, values == 2)).astype("float32")
    return image, truth


with zipfile.ZipFile("data/raw/RIM-ONE-r3.zip") as rim, zipfile.ZipFile("data/raw/ORIGA_candidate_figshare.zip") as combined, torch.no_grad():
    rows = []
    for row in split:
        if row["split"] != "test":
            continue
        image, truth = load_record(row)
        tensor = torch.from_numpy(
            np.asarray(image, dtype=np.uint8).transpose(2, 0, 1).copy()
        )[None].float() / 255
        pred = model(tensor).sigmoid()[0].numpy() > 0.5
        result = {
            "dataset": row["dataset"],
            "image": row["image"],
        }
        for i, region in enumerate(("disc", "cup")):
            p = pred[i]
            y = truth[i] > 0
            intersection = int(np.logical_and(p, y).sum())
            predicted = int(p.sum())
            actual = int(y.sum())
            union = int(np.logical_or(p, y).sum())
            result[f"{region}_dice"] = (2 * intersection + 1e-6) / (predicted + actual + 1e-6)
            result[f"{region}_iou"] = (intersection + 1e-6) / (union + 1e-6)
        rows.append(result)

    destination = root / "test_predictions.csv"
    with destination.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

    index = []
    for dataset in ("RIM-ONE r3", "ORIGA", "REFUGE"):
        subset = [row for row in rows if row["dataset"] == dataset]
        ordered = sorted(subset, key=lambda r: (r["disc_dice"] + r["cup_dice"]) / 2)
        selected = [("worst", ordered[0]), ("best", ordered[-1])]
        for quality, result in selected:
            row = next(r for r in split if r["dataset"] == dataset and r["image"] == result["image"])
            image, truth = load_record(row)
            tensor = torch.from_numpy(
                np.asarray(image, dtype=np.uint8).transpose(2, 0, 1).copy()
            )[None].float() / 255
            pred = model(tensor).sigmoid()[0].numpy() > 0.5
            panel = Image.new("RGB", (768, 256), "white")
            panel.paste(image, (0, 0))
            for panel_x, region_index in ((256, 0), (512, 1)):
                region = truth[region_index] > 0
                predicted = pred[region_index]
                overlay = np.asarray(image).copy()
                overlay[region & predicted] = 0.55 * overlay[region & predicted] + 0.45 * np.array([0, 220, 110])
                overlay[region & ~predicted] = 0.55 * overlay[region & ~predicted] + 0.45 * np.array([50, 120, 255])
                overlay[predicted & ~region] = 0.55 * overlay[predicted & ~region] + 0.45 * np.array([255, 170, 40])
                panel.paste(Image.fromarray(overlay.astype("uint8")), (panel_x, 0))
            draw = ImageDraw.Draw(panel)
            draw.text((8, 238), f"{dataset} {quality} original", fill="black")
            draw.text((264, 238), "green overlap / blue FN / orange FP: disc", fill="black")
            draw.text((520, 238), "same encoding: cup", fill="black")
            path = out / f"{quality}_{dataset.lower().replace(' ', '_').replace('-', '_')}_{Path(result['image']).stem}.png"
            panel.save(path)
            index.append({
                "dataset": dataset,
                "quality": quality,
                "image": result["image"],
                "panel": str(path).replace("\\", "/"),
                "disc_dice": result["disc_dice"],
                "cup_dice": result["cup_dice"],
            })

(out / "index.json").write_text(json.dumps(index, indent=2), encoding="utf-8")
print(f"Wrote {len(rows)} per-image scores and {len(index)} pooled segmentation panels")


In [ ]:
%%writefile tools/train_drishti_segmentation.py
"""Train a compact DRISHTI-GS1 Kaggle-mirror optic disc/cup segmentation baseline."""
from __future__ import annotations

import csv
import io
import json
import random
import sys
import zipfile
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from models.unet_small import UNet


SEED = 42
SIZE = 256
EPOCHS = 8
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(4)

archive = zipfile.ZipFile("data/provisional/drishti_gs1/drishti_gs1_kaggle_mirror.zip")
rows = list(csv.DictReader(Path("data/audit/drishti_gs1_kaggle_mirror_manifest.csv").open(encoding="utf-8")))
train_rows = [row for row in rows if row["official_style_split"] == "train"]
test_rows = [row for row in rows if row["official_style_split"] == "test"]
train_idx, val_idx = train_test_split(
    np.arange(len(train_rows)),
    test_size=0.2,
    stratify=[row["label"] for row in train_rows],
    random_state=SEED,
)
train_pool = train_rows
train_rows = [train_pool[i] for i in train_idx]
val_rows = [train_pool[i] for i in val_idx]

root = Path("outputs/drishti_gs1_mirror")
root.mkdir(parents=True, exist_ok=True)
split_rows = (
    [{"image": row["image"], "label": row["label"], "split": "train"} for row in train_rows]
    + [{"image": row["image"], "label": row["label"], "split": "val"} for row in val_rows]
    + [{"image": row["image"], "label": row["label"], "split": "test"} for row in test_rows]
)
with (root / "split.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=["image", "label", "split"])
    writer.writeheader()
    writer.writerows(split_rows)


class Drishti(Dataset):
    def __init__(self, records, augment=False):
        self.records = records
        self.augment = augment

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        row = self.records[i]
        with Image.open(io.BytesIO(archive.read(row["image"]))).convert("RGB") as image:
            image = image.resize((SIZE, SIZE), Image.Resampling.BICUBIC)
        masks = []
        for key in ("disc_mask", "cup_mask"):
            with Image.open(io.BytesIO(archive.read(row[key]))).convert("L") as mask:
                mask = mask.resize((SIZE, SIZE), Image.Resampling.BILINEAR)
                masks.append((np.asarray(mask) >= 128).astype("float32"))
        pixels = np.asarray(image).copy()
        truth = np.stack(masks)
        if self.augment and random.random() < 0.5:
            pixels = pixels[:, ::-1].copy()
            truth = truth[:, :, ::-1].copy()
        return (
            torch.from_numpy(np.ascontiguousarray(pixels.transpose(2, 0, 1))).float() / 255,
            torch.from_numpy(truth),
        )


train_ds = Drishti(train_rows, augment=True)
val_ds = Drishti(val_rows)
test_ds = Drishti(test_rows)
model = UNet()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
bce = nn.BCEWithLogitsLoss()
history = []
best = -1.0


def scores(dataset):
    model.eval()
    totals = np.zeros((2, 2), dtype=np.float64)
    count = 0
    with torch.no_grad():
        for x, y in DataLoader(dataset, batch_size=4):
            pred = (model(x).sigmoid() > 0.5).numpy()
            truth = y.numpy() > 0
            for channel in range(2):
                p = pred[:, channel]
                g = truth[:, channel]
                inter = (p & g).sum(axis=(1, 2))
                ps = p.sum(axis=(1, 2))
                gs = g.sum(axis=(1, 2))
                union = (p | g).sum(axis=(1, 2))
                totals[channel, 0] += ((2 * inter + 1e-6) / (ps + gs + 1e-6)).sum()
                totals[channel, 1] += ((inter + 1e-6) / (union + 1e-6)).sum()
            count += len(x)
    return totals / count


for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    for x, y in DataLoader(train_ds, batch_size=4, shuffle=True):
        logits = model(x)
        prob = logits.sigmoid()
        dice = 1 - ((2 * (prob * y).sum((0, 2, 3)) + 1) / ((prob + y).sum((0, 2, 3)) + 1)).mean()
        loss = bce(logits, y) + dice
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    val = scores(val_ds)
    rank = float(val[:, 0].mean())
    history.append({
        "epoch": epoch,
        "train_loss": float(np.mean(losses)),
        "val_disc_dice": float(val[0, 0]),
        "val_cup_dice": float(val[1, 0]),
    })
    if rank > best:
        best = rank
        torch.save(model.state_dict(), root / "best.pt")
    print(history[-1], flush=True)

model.load_state_dict(torch.load(root / "best.pt", map_location="cpu", weights_only=True))
test = scores(test_ds)
metrics = {
    "status": "drishti_gs1_kaggle_mirror_partial_segmentation",
    "dataset": "DRISHTI-GS1",
    "source": "lokeshsaipureddi Kaggle mirror",
    "architecture": "compact randomly initialized UNet",
    "seed": SEED,
    "epochs": EPOCHS,
    "n_train": len(train_rows),
    "n_val": len(val_rows),
    "n_test": len(test_rows),
    "test_disc_dice": float(test[0, 0]),
    "test_disc_iou": float(test[0, 1]),
    "test_cup_dice": float(test[1, 0]),
    "test_cup_iou": float(test[1, 1]),
    "limitations": "Third-party Kaggle mirror; test soft masks used only for held-out evaluation; official registration not used.",
}
(root / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
(root / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
print(json.dumps(metrics, indent=2), flush=True)
archive.close()


In [ ]:
%%writefile tools/evaluate_drishti_mirror.py
"""Save held-out DRISHTI-GS1 mirror segmentation scores and visual panels."""
from __future__ import annotations

import csv
import io
import json
import sys
import zipfile
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageDraw

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from models.unet_small import UNet


SIZE = 256
root = Path("outputs/drishti_gs1_mirror")
archive = zipfile.ZipFile("data/provisional/drishti_gs1/drishti_gs1_kaggle_mirror.zip")
manifest = list(csv.DictReader(Path("data/audit/drishti_gs1_kaggle_mirror_manifest.csv").open(encoding="utf-8")))
split = list(csv.DictReader((root / "split.csv").open(encoding="utf-8")))
test_names = {row["image"] for row in split if row["split"] == "test"}
lookup = {row["image"]: row for row in manifest}
model = UNet()
model.load_state_dict(torch.load(root / "best.pt", map_location="cpu", weights_only=True))
model.eval()

rows = []
with torch.no_grad():
    for name in sorted(test_names):
        record = lookup[name]
        with Image.open(io.BytesIO(archive.read(name))).convert("RGB") as image:
            image = image.resize((SIZE, SIZE), Image.Resampling.BICUBIC)
        masks = []
        for key in ("disc_mask", "cup_mask"):
            with Image.open(io.BytesIO(archive.read(record[key]))).convert("L") as mask:
                mask = mask.resize((SIZE, SIZE), Image.Resampling.BILINEAR)
                masks.append((np.asarray(mask) >= 128))
        pixels = np.asarray(image).copy()
        tensor = torch.from_numpy(pixels.transpose(2, 0, 1).copy())[None].float() / 255
        pred = model(tensor).sigmoid()[0].detach().numpy() > 0.5
        row = {"dataset": "DRISHTI-GS1", "source": "lokeshsaipureddi Kaggle mirror", "image": name}
        for i, region in enumerate(("disc", "cup")):
            p, y = pred[i], masks[i]
            inter = int(np.logical_and(p, y).sum())
            ps = int(p.sum())
            gs = int(y.sum())
            union = int(np.logical_or(p, y).sum())
            row[f"{region}_dice"] = (2 * inter + 1e-6) / (ps + gs + 1e-6)
            row[f"{region}_iou"] = (inter + 1e-6) / (union + 1e-6)
        rows.append(row)

with (root / "test_predictions.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

out = root / "examples"
out.mkdir(parents=True, exist_ok=True)
index = []
ordered = sorted(rows, key=lambda r: (r["disc_dice"] + r["cup_dice"]) / 2)
for quality, record in (("worst", ordered[0]), ("best", ordered[-1])):
    row = lookup[record["image"]]
    with Image.open(io.BytesIO(archive.read(row["image"]))).convert("RGB") as image:
        image = image.resize((SIZE, SIZE), Image.Resampling.BICUBIC)
    truth = []
    for key in ("disc_mask", "cup_mask"):
        with Image.open(io.BytesIO(archive.read(row[key]))).convert("L") as mask:
            mask = mask.resize((SIZE, SIZE), Image.Resampling.BILINEAR)
            truth.append((np.asarray(mask) >= 128))
    pixels = np.asarray(image).copy()
    tensor = torch.from_numpy(pixels.transpose(2, 0, 1).copy())[None].float() / 255
    pred = model(tensor).sigmoid()[0].detach().numpy() > 0.5
    panel = Image.new("RGB", (768, 256), "white")
    panel.paste(image, (0, 0))
    for panel_x, i in ((256, 0), (512, 1)):
        y = truth[i]
        p = pred[i]
        overlay = pixels.copy()
        overlap = y & p
        false_negative = y & ~p
        false_positive = p & ~y
        overlay[overlap] = 0.55 * overlay[overlap] + 0.45 * np.array([0, 220, 110])
        overlay[false_negative] = 0.55 * overlay[false_negative] + 0.45 * np.array([50, 120, 255])
        overlay[false_positive] = 0.55 * overlay[false_positive] + 0.45 * np.array([255, 170, 40])
        panel.paste(Image.fromarray(overlay.astype("uint8")), (panel_x, 0))
    draw = ImageDraw.Draw(panel)
    draw.text((8, 238), f"{quality} original", fill="black")
    draw.text((264, 238), "disc: green overlap / blue FN / orange FP", fill="black")
    draw.text((520, 238), "cup: same encoding", fill="black")
    path = out / f"{quality}_{Path(row['image']).stem}.png"
    panel.save(path)
    index.append({"quality": quality, "image": row["image"], "panel": str(path).replace("\\", "/"),
                  "disc_dice": record["disc_dice"], "cup_dice": record["cup_dice"]})

(out / "index.json").write_text(json.dumps(index, indent=2), encoding="utf-8")
archive.close()
print(json.dumps({"n_test": len(rows), "panels": index}, indent=2))


In [ ]:
if RUN_POOLED_SEGMENTER:
    run_script('train_pooled_segmentation.py', 'outputs/pooled_segmentation/metrics.json')
    run_script('render_pooled_segmentation.py', 'outputs/pooled_segmentation/test_predictions.csv')
    run_script('train_drishti_segmentation.py', 'outputs/drishti_gs1_mirror/metrics.json')
    run_script('evaluate_drishti_mirror.py', 'outputs/drishti_gs1_mirror/test_predictions.csv')
for path in ('outputs/pooled_segmentation/metrics.json', 'outputs/drishti_gs1_mirror/metrics.json'):
    result = json.loads((ROOT / path).read_text())
    print(path, json.dumps(result.get('test_metrics', result), indent=2)[:1500])
print('Step 5 complete: segmentation metrics and visual examples saved.')

## Step 6 — Matched whole-image versus predicted-disc crop

The segmentation model predicts disc locations for **train, validation, and test** classification images. The crop model never uses a test ground-truth mask. Compare both ResNet-18 feature classifiers on exactly the same RIM-ONE test images, with the same candidate regularization values.

Run the source cells below, then the stage runner. Every file they create comes from code **inside this notebook**.

In [ ]:
%%writefile tools/train_rim_whole_vs_crop.py
"""Matched whole-image vs predicted-disc-crop ResNet-18 comparison on RIM-ONE r3."""
from __future__ import annotations

import csv
import io
import json
import random
import sys
import zipfile
from pathlib import Path

import joblib
import numpy as np
import torch
from PIL import Image, ImageOps
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support, roc_auc_score
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.transforms import Normalize, ToTensor

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from models.unet_small import UNet


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(4)

root = Path("outputs/rim_whole_vs_crop")
root.mkdir(parents=True, exist_ok=True)
manifest = {
    row["image"]: row
    for row in csv.DictReader(Path("data/audit/rim_one_r3_manifest.csv").open(encoding="utf-8"))
}
seg_split = list(csv.DictReader(Path("outputs/rim_unet_baseline/split.csv").open(encoding="utf-8")))
binary = [
    row for row in seg_split
    if manifest[row["image"]]["label_original"] in ("normal", "glaucoma")
]

segmenter = UNet()
segmenter.load_state_dict(torch.load(
    "outputs/rim_unet_baseline/best.pt", map_location="cpu", weights_only=True
))
segmenter.eval()

feature_model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
feature_model.fc = torch.nn.Identity()
feature_model.eval()
to_tensor = ToTensor()
normalize = Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])


def left_image(archive: zipfile.ZipFile, record: dict) -> Image.Image:
    with Image.open(io.BytesIO(archive.read(record["image"]))).convert("RGB") as image:
        left = image.crop((0, 0, image.width // 2, image.height)).resize((256, 256), Image.Resampling.BICUBIC)
    return left


def predicted_disc(image: Image.Image) -> np.ndarray:
    tensor = torch.from_numpy(
        np.asarray(image, dtype=np.uint8).transpose(2, 0, 1).copy()
    )[None].float() / 255
    with torch.no_grad():
        logits = segmenter(tensor)
    return logits.sigmoid()[0, 0].numpy() > 0.5


def crop_disc(image: Image.Image, disc: np.ndarray) -> Image.Image:
    ys, xs = np.where(disc)
    if len(xs) == 0:
        return ImageOps.pad(image, (224, 224), method=Image.Resampling.BICUBIC, color=(0, 0, 0))
    x0, x1 = int(xs.min()), int(xs.max())
    y0, y1 = int(ys.min()), int(ys.max())
    size = int(max(x1 - x0, y1 - y0) * 1.5)
    cx, cy = (x0 + x1) // 2, (y0 + y1) // 2
    box = (
        max(0, cx - size // 2),
        max(0, cy - size // 2),
        min(image.width, cx + (size + 1) // 2),
        min(image.height, cy + (size + 1) // 2),
    )
    cropped = image.crop(box)
    return ImageOps.pad(cropped, (224, 224), method=Image.Resampling.BICUBIC, color=(0, 0, 0))


def whole_image(image: Image.Image) -> Image.Image:
    return ImageOps.pad(image, (224, 224), method=Image.Resampling.BICUBIC, color=(0, 0, 0))


X = {"whole": [], "crop": []}
labels = []
splits = []
members = []

with zipfile.ZipFile("data/raw/RIM-ONE-r3.zip") as archive, torch.no_grad():
    for row in binary:
        record = manifest[row["image"]]
        image = left_image(archive, record)
        disc = predicted_disc(image)
        variants = {
            "whole": whole_image(image),
            "crop": crop_disc(image, disc),
        }
        for name, prepared in variants.items():
            tensor = normalize(to_tensor(prepared))[None]
            feature = feature_model(tensor).numpy()[0]
            X[name].append(feature)
        labels.append(record["label_original"] == "glaucoma")
        splits.append(row["split"])
        members.append(row["image"])

X = {name: np.asarray(values) for name, values in X.items()}
labels = np.asarray(labels, dtype=int)
splits = np.asarray(splits)
train = np.where(splits == "train")[0]
val = np.where(splits == "val")[0]
trainval = np.where(splits != "test")[0]
test = np.where(splits == "test")[0]

comparison = {"dataset": "RIM-ONE r3", "seed": SEED, "n_train": len(train),
              "n_val": len(val), "n_test": len(test), "protocol": {
                  "whole": "left stereo fundus view letterboxed to 224x224",
                  "crop": "disc crop from saved U-Net predicted disc mask in every split; no classification ground-truth mask crop",
              }, "selection_metric": "validation AUC", "candidates": (0.01, 0.1, 1.0, 10.0)}

for name in ("whole", "crop"):
    trials = []
    for c in comparison["candidates"]:
        classifier = LogisticRegression(C=c, max_iter=1000, random_state=SEED)
        classifier.fit(X[name][train], labels[train])
        probabilities = classifier.predict_proba(X[name][val])[:, 1]
        trials.append({"C": c, "val_auc": float(roc_auc_score(labels[val], probabilities))})
    best_c = max(trials, key=lambda item: item["val_auc"])["C"]
    classifier = LogisticRegression(C=best_c, max_iter=1000, random_state=SEED)
    classifier.fit(X[name][trainval], labels[trainval])
    joblib.dump(classifier, root / f"{name}_classifier.joblib")

    probabilities = classifier.predict_proba(X[name][test])[:, 1]
    predicted = (probabilities >= 0.5).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels[test], predicted, average="binary", zero_division=0
    )
    tn, fp, fn, tp = confusion_matrix(labels[test], predicted, labels=[0, 1]).ravel()
    metrics = {
        "n_test": len(test),
        "accuracy": float(accuracy_score(labels[test], predicted)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "auc": float(roc_auc_score(labels[test], probabilities)),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "selected_C": best_c,
        "val_trials": trials,
    }
    comparison[name] = metrics
    with (root / f"{name}_test_predictions.csv").open("w", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        writer.writerow(["dataset", "member", "true_label", "probability_glaucoma", "predicted_label"])
        writer.writerows([
            ("RIM-ONE r3", members[i], int(labels[i]), float(probability), int(yhat))
            for i, probability, yhat in zip(test, probabilities, predicted)
        ])

with (root / "metrics.json").open("w", encoding="utf-8") as stream:
    json.dump(comparison, stream, indent=2)
with (root / "split.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=["member", "label", "split"])
    writer.writeheader()
    writer.writerows({"member": members[i], "label": int(labels[i]), "split": splits[i]} for i in range(len(members)))

print(json.dumps(comparison, indent=2))


In [ ]:
%%writefile tools/render_rim_crop_comparison.py
"""Render the measured RIM-ONE whole-image versus disc-crop comparison."""
from pathlib import Path
import json

import matplotlib.pyplot as plt


root = Path("outputs/rim_whole_vs_crop")
metrics = json.loads((root / "metrics.json").read_text(encoding="utf-8"))
names = ("whole", "crop")
labels = ["Whole image", "Predicted disc crop"]
auc = [metrics[name]["auc"] for name in names]
accuracy = [metrics[name]["accuracy"] for name in names]

fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.8), dpi=160)
for ax, values, title, ylabel in (
    (axes[0], auc, "Held-out ROC AUC", "AUC"),
    (axes[1], accuracy, "Held-out accuracy", "Accuracy"),
):
    bars = ax.bar(labels, values, color=["#4c78a8", "#f58518"])
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 1)
    ax.grid(axis="y", alpha=0.2)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, value + 0.02,
                f"{value:.3f}", ha="center", va="bottom", fontsize=9)
fig.suptitle("RIM-ONE r3 matched split: whole image vs disc crop")
fig.tight_layout()
output = root / "comparison.png"
fig.savefig(output, bbox_inches="tight")
plt.close(fig)
print(output)


In [ ]:
if RUN_BASELINES:
    run_script('train_rim_whole_vs_crop.py', 'outputs/rim_whole_vs_crop/metrics.json')
    run_script('render_rim_crop_comparison.py', 'outputs/rim_whole_vs_crop/comparison.png')
print(json.loads((ROOT / 'outputs/rim_whole_vs_crop/metrics.json').read_text())['protocol'])
display(Image.open(ROOT / 'outputs/rim_whole_vs_crop/comparison.png'))
print('Step 6 complete: matched crop experiment saved.')

## Step 7 — DRISHTI mirror classification and pooled fine-tuning

Train a separate DRISHTI mirror classifier and fine-tune a pooled ResNet-18 across all five source names. The mirror provenance remains visible in every result. The fine-tuned model uses ImageNet initialization, dataset/class-balanced sampling, a fixed seed, saved validation history, and held-out test predictions. Use a free Colab GPU for this stage.

Run the source cells below, then the stage runner. Every file they create comes from code **inside this notebook**.

In [ ]:
%%writefile tools/train_drishti_classifier.py
"""Frozen ResNet-18 classifier baseline on the Kaggle DRISHTI-GS1 mirror."""
from __future__ import annotations

import csv
import io
import json
import zipfile
from pathlib import Path

import joblib
import numpy as np
import torch
from PIL import Image, ImageOps
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.transforms import Normalize, ToTensor


SEED = 42
root = Path("outputs/drishti_gs1_mirror_classifier")
root.mkdir(parents=True, exist_ok=True)
archive = zipfile.ZipFile("data/provisional/drishti_gs1/drishti_gs1_kaggle_mirror.zip")
rows = list(csv.DictReader(Path("data/audit/drishti_gs1_kaggle_mirror_manifest.csv").open(encoding="utf-8")))
train_pool = [row for row in rows if row["official_style_split"] == "train"]
test_rows = [row for row in rows if row["official_style_split"] == "test"]
train_idx, val_idx = train_test_split(
    np.arange(len(train_pool)),
    test_size=0.2,
    stratify=[row["label"] for row in train_pool],
    random_state=SEED,
)
train_rows = [train_pool[i] for i in train_idx]
val_rows = [train_pool[i] for i in val_idx]

model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = torch.nn.Identity()
model.eval()
to_tensor = ToTensor()
normalize = Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])


def features(records):
    values = []
    with torch.no_grad():
        for start in range(0, len(records), 8):
            batch = []
            for record in records[start:start + 8]:
                with Image.open(io.BytesIO(archive.read(record["image"]))).convert("RGB") as image:
                    image = ImageOps.pad(image, (224, 224), method=Image.Resampling.BICUBIC, color=(0, 0, 0))
                    batch.append(normalize(to_tensor(image)))
            values.append(model(torch.stack(batch)).numpy())
    return np.concatenate(values)


X_train = features(train_rows)
X_val = features(val_rows)
X_trainval = features(train_rows + val_rows)
X_test = features(test_rows)
y_train = np.asarray([row["label"] == "glaucoma" for row in train_rows], dtype=int)
y_val = np.asarray([row["label"] == "glaucoma" for row in val_rows], dtype=int)
y_trainval = np.asarray([row["label"] == "glaucoma" for row in train_rows + val_rows], dtype=int)
y_test = np.asarray([row["label"] == "glaucoma" for row in test_rows], dtype=int)

trials = []
for c in (0.01, 0.1, 1.0, 10.0):
    classifier = LogisticRegression(C=c, max_iter=1000, random_state=SEED)
    classifier.fit(X_train, y_train)
    probability = classifier.predict_proba(X_val)[:, 1]
    trials.append({"C": c, "val_auc": float(roc_auc_score(y_val, probability))})
selected = max(trials, key=lambda item: item["val_auc"])["C"]
classifier = LogisticRegression(C=selected, max_iter=1000, random_state=SEED)
classifier.fit(X_trainval, y_trainval)
joblib.dump(classifier, root / "classifier.joblib")

probability = classifier.predict_proba(X_test)[:, 1]
predicted = (probability >= 0.5).astype(int)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, predicted, average="binary", zero_division=0)
tn, fp, fn, tp = confusion_matrix(y_test, predicted, labels=[0, 1]).ravel()
metrics = {
    "status": "drishti_gs1_kaggle_mirror_partial_classification",
    "dataset": "DRISHTI-GS1",
    "source": "lokeshsaipureddi Kaggle mirror",
    "architecture": "frozen ImageNet ResNet-18 features + logistic head",
    "seed": SEED,
    "n_train": len(train_rows),
    "n_val": len(val_rows),
    "n_test": len(test_rows),
    "accuracy": float(accuracy_score(y_test, predicted)),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "auc": float(roc_auc_score(y_test, probability)),
    "TN": int(tn),
    "FP": int(fp),
    "FN": int(fn),
    "TP": int(tp),
    "selected_C": selected,
    "val_trials": trials,
}
with (root / "metrics.json").open("w", encoding="utf-8") as stream:
    json.dump(metrics, stream, indent=2)
with (root / "test_predictions.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(["dataset", "member", "true_label", "probability_glaucoma", "predicted_label"])
    writer.writerows([
        ("DRISHTI-GS1", row["image"], int(y), float(p), int(yhat))
        for row, y, p, yhat in zip(test_rows, y_test, probability, predicted)
    ])
split = [{"image": row["image"], "label": row["label"], "split": "train"} for row in train_rows]
split += [{"image": row["image"], "label": row["label"], "split": "val"} for row in val_rows]
split += [{"image": row["image"], "label": row["label"], "split": "test"} for row in test_rows]
with (root / "split.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=["image", "label", "split"])
    writer.writeheader()
    writer.writerows(split)

print(json.dumps(metrics, indent=2))
archive.close()


In [ ]:
%%writefile tools/train_pooled_finetuned_classifier.py
"""Fine-tune a pooled glaucoma ResNet-18 across ACRIMA, RIM-ONE, ORIGA, REFUGE, and DRISHTI-GS1."""
from __future__ import annotations

import csv
import io
import json
import random
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support, roc_auc_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.models import ResNet18_Weights, resnet18
from torchvision.transforms import Normalize, ToTensor


SEED = 42
EPOCHS = 12
BATCH_SIZE = 32
SIZE = 224
LR = 3e-4
WEIGHT_DECAY = 1e-4
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

root = Path("outputs/pooled_finetuned_classifier")
root.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(max(1, min(8, torch.get_num_threads())))

records = []


def add(dataset, member, label, split, archive_kind="acrima", drishti_label=None, source_split=""):
    records.append({
        "dataset": dataset,
        "member": member,
        "label": int(label),
        "split": split,
        "archive": archive_kind,
        "drishti_label": drishti_label,
        "source_split": source_split,
    })


for row in csv.DictReader(Path("outputs/acrima_resnet18_frozen/split.csv").open(encoding="utf-8")):
    add("ACRIMA", row["member"], row["label"] == "glaucoma", row["split"], "acrima")

for row in csv.DictReader(Path("outputs/rim_unet_baseline/split.csv").open(encoding="utf-8")):
    rim_labels = {r["image"]: r["label_original"] for r in csv.DictReader(Path("data/audit/rim_one_r3_manifest.csv").open(encoding="utf-8"))}
    label = rim_labels[row["image"]]
    if label in ("normal", "glaucoma"):
        add("RIM-ONE r3", row["image"], label == "glaucoma", row["split"], "rim")

origa = list(csv.DictReader(Path("data/audit/origa_manifest.csv").open(encoding="utf-8")))
origa_a = [row for row in origa if row["provided_set"] == "A"]
origa_b = [row for row in origa if row["provided_set"] == "B"]
origa_train_idx, origa_val_idx = train_test_split(
    np.arange(len(origa_a)),
    test_size=0.2,
    stratify=[row["label"] for row in origa_a],
    random_state=SEED,
)
for i, row in enumerate(origa_a):
    add("ORIGA", row["image"], int(row["label"]), "val" if i in set(origa_val_idx) else "train", "combined")
for row in origa_b:
    add("ORIGA", row["image"], int(row["label"]), "test", "combined")

refuge = list(csv.DictReader(Path("data/audit/refuge_manifest.csv").open(encoding="utf-8")))
refuge_train = [row for row in refuge if row["split"] == "train"]
refuge_train_idx, refuge_val_idx = train_test_split(
    np.arange(len(refuge_train)),
    test_size=0.2,
    stratify=[row["label"] for row in refuge_train],
    random_state=SEED,
)
for i, row in enumerate(refuge_train):
    add("REFUGE", row["image"], int(row["label"]), "val" if i in set(refuge_val_idx) else "train", "combined")
for row in refuge:
    if row["split"] in ("val", "test"):
        add("REFUGE", row["image"], int(row["label"]), "test", "combined", source_split=row["split"])

drishti = list(csv.DictReader(Path("data/audit/drishti_gs1_kaggle_mirror_manifest.csv").open(encoding="utf-8")))
drishti_train = [row for row in drishti if row["official_style_split"] == "train"]
drishti_test = [row for row in drishti if row["official_style_split"] == "test"]
drishti_train_idx, drishti_val_idx = train_test_split(
    np.arange(len(drishti_train)),
    test_size=0.2,
    stratify=[row["label"] for row in drishti_train],
    random_state=SEED,
)
for i, row in enumerate(drishti_train):
    split = "val" if i in set(drishti_val_idx) else "train"
    add("DRISHTI-GS1", row["image"], row["label"] == "glaucoma", split, "drishti", row["label"])
for row in drishti_test:
    add("DRISHTI-GS1", row["image"], row["label"] == "glaucoma", "test", "drishti", row["label"])

with (root / "split.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(records[0]))
    writer.writeheader()
    writer.writerows(records)


class PooledDataset(Dataset):
    def __init__(self, indices, augment=False):
        self.indices = indices
        self.augment = augment
        self.archives = {
            "acrima": zipfile.ZipFile("data/raw/ACRIMA_Database.zip"),
            "rim": zipfile.ZipFile("data/raw/RIM-ONE-r3.zip"),
            "combined": zipfile.ZipFile("data/raw/ORIGA_candidate_figshare.zip"),
            "drishti": zipfile.ZipFile("data/provisional/drishti_gs1/drishti_gs1_kaggle_mirror.zip"),
        }

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        record = records[self.indices[i]]
        raw = self.archives[record["archive"]].read(record["member"])
        with Image.open(io.BytesIO(raw)).convert("RGB") as image:
            if record["archive"] == "rim":
                image = image.crop((0, 0, image.width // 2, image.height))
            image = ImageOps.pad(image, (SIZE, SIZE), method=Image.Resampling.BICUBIC, color=(0, 0, 0))
            pixels = np.asarray(image).copy()
        if self.augment and random.random() < 0.5:
            pixels = pixels[:, ::-1].copy()
        tensor = torch.from_numpy(np.ascontiguousarray(pixels.transpose(2, 0, 1))).float() / 255
        return normalize(tensor), torch.tensor(record["label"], dtype=torch.long)

    def close(self):
        for archive in self.archives.values():
            archive.close()


weights = ResNet18_Weights.DEFAULT
normalize = Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
model = resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

train_indices = [i for i, row in enumerate(records) if row["split"] == "train"]
val_indices = [i for i, row in enumerate(records) if row["split"] == "val"]
test_indices = [i for i, row in enumerate(records) if row["split"] == "test"]

counts = Counter((row["dataset"], row["label"]) for i, row in enumerate(records) if i in train_indices)
dataset_class_weights = {key: len(train_indices) / (len(counts) * count) for key, count in counts.items()}
sample_weights = [dataset_class_weights[(records[i]["dataset"], records[i]["label"])] for i in train_indices]
sampler = torch.utils.data.WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_ds = PooledDataset(train_indices, augment=True)
val_ds = PooledDataset(val_indices)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = []
best_val_auc = -1.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    for pixels, labels in train_loader:
        pixels, labels = pixels.to(device), labels.to(device)
        logits = model(pixels)
        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    model.eval()
    probabilities = []
    truths = []
    datasets = []
    with torch.no_grad():
        for pixels, labels in val_loader:
            logits = model(pixels.to(device))
            probabilities.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
            truths.append(labels.numpy())
    probability = np.concatenate(probabilities)
    truth = np.concatenate(truths)
    auc = float(roc_auc_score(truth, probability)) if len(np.unique(truth)) == 2 else float("nan")
    history.append({"epoch": epoch, "train_loss": float(np.mean(losses)), "val_auc": auc})
    if auc > best_val_auc:
        best_val_auc = auc
        torch.save({
            "model_state": model.state_dict(),
            "weights": "ResNet18_Weights.DEFAULT",
            "epoch": epoch,
            "seed": SEED,
        }, root / "best.pt")
    scheduler.step()
    print(history[-1], flush=True)

checkpoint = torch.load(root / "best.pt", map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state"])
model.eval()

test_ds = PooledDataset(test_indices)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
probability_groups = []
truth_groups = []
with torch.no_grad():
    for pixels, labels in test_loader:
        logits = model(pixels.to(device))
        probability_groups.append(torch.softmax(logits, dim=1)[:, 1].cpu().numpy())
        truth_groups.append(labels.numpy())
probabilities = np.concatenate(probability_groups)
truths = np.concatenate(truth_groups)

test_rows = [records[i] for i in test_indices]
def score_positions(dataset_positions):
    dataset_rows = [test_rows[i] for i in dataset_positions]
    y = np.asarray([row["label"] for row in dataset_rows])
    p = probabilities[dataset_positions]
    predicted = (p >= 0.5).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(y, predicted, average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, predicted, labels=[0, 1]).ravel()
    return {
        "n_test": len(dataset_rows),
        "accuracy": float(accuracy_score(y, predicted)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "auc": float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else None,
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
    }


metrics = {}
for dataset in sorted(set(row["dataset"] for row in test_rows)):
    metrics[dataset] = score_positions([i for i, row in enumerate(test_rows) if row["dataset"] == dataset])
refuge_by_source_split = {
    split: score_positions([i for i, row in enumerate(test_rows) if row["dataset"] == "REFUGE" and row["source_split"] == split])
    for split in ("val", "test")
}

summary = {
    "status": "five_source_mirror_finetuned_partial",
    "architecture": "fine-tuned ResNet-18",
    "device": str(device),
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "protocol": "ACRIMA fixed split; RIM-ONE fixed split excluding suspects; ORIGA A/B split; REFUGE supplied train/val/test; DRISHTI Kaggle mirror 40/10/51",
    "history": history,
    "test_metrics": metrics,
    "refuge_test_metrics_by_source_split": refuge_by_source_split,
    "limitations": "DRISHTI-GS1 is a Kaggle mirror; ORIGA and REFUGE provenance caveats remain; final source policy not yet frozen.",
}
(root / "metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

with (root / "test_predictions.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(["dataset", "member", "source_split", "true_label", "probability_glaucoma", "predicted_label"])
    current = 0
    for row, probability, truth, predicted in zip(
        [records[i] for i in test_indices], probabilities, truths, (probabilities >= 0.5).astype(int)
    ):
        writer.writerow([row["dataset"], row["member"], row["source_split"], int(truth), float(probability), int(predicted)])

train_ds.close()
val_ds.close()
test_ds.close()
print(json.dumps(summary, indent=2), flush=True)


In [ ]:
if RUN_FINETUNING:
    if DEVICE.type != 'cuda':
        raise RuntimeError('Choose Runtime > Change runtime type > GPU in Colab, then rerun Step 0 and this stage.')
    run_script('train_drishti_classifier.py', 'outputs/drishti_gs1_mirror_classifier/metrics.json')
    run_script('train_pooled_finetuned_classifier.py', 'outputs/pooled_finetuned_classifier/metrics.json')
fine = json.loads((ROOT / 'outputs/pooled_finetuned_classifier/metrics.json').read_text())
print('Fine-tuning device:', fine['device'])
for dataset, metrics in fine['test_metrics'].items():
    print(dataset, 'n=', metrics['n_test'], 'AUC=', metrics['auc'], 'F1=', metrics['f1'])
print('Step 7 complete: trained weights and held-out predictions saved.')

## Step 8 — XAI, architecture, and figures

Generate predicted-class Grad-CAM panels, segmentation examples, confusion/ROC figures, and a baseline architecture diagram. A heatmap shows the model's sensitivity; it does not prove clinical reasoning.

Run the source cells below, then the stage runner. Every file they create comes from code **inside this notebook**.

In [ ]:
%%writefile tools/render_gradcam.py
"""Grad-CAM for the measured frozen ResNet-18 + logistic classifier."""
import csv
import io
import zipfile
from pathlib import Path

import joblib
import numpy as np
import torch
from PIL import Image, ImageDraw
from torchvision.models import ResNet18_Weights, resnet18

torch.set_num_threads(4)
root=Path('outputs/acrima_resnet18_frozen'); out=root/'gradcam'; out.mkdir(exist_ok=True)
manifest={r['image_id']:r for r in csv.DictReader(Path('data/audit/acrima_manifest.csv').open(encoding='utf-8'))}
predictions=list(csv.DictReader((root/'test_predictions.csv').open(encoding='utf-8')))
groups={'true_positive':[], 'true_negative':[], 'false_positive':[], 'false_negative':[]}
for r in predictions:
    truth=int(r['true_label']); pred=int(r['predicted_label'])
    key=('true_' if truth==pred else 'false_')+('positive' if pred else 'negative')
    groups[key].append(r)
chosen=[r for key in groups for r in groups[key][:2]]
clf=joblib.load(root/'classifier.joblib')
model=resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc=torch.nn.Linear(512,1)
with torch.no_grad():
    model.fc.weight.copy_(torch.from_numpy(clf.coef_).float()); model.fc.bias.copy_(torch.from_numpy(clf.intercept_).float())
model.eval(); transform=ResNet18_Weights.IMAGENET1K_V1.transforms()
acts=[]; grads=[]
def capture(_m,_i,o):
    acts.append(o)
    o.register_hook(lambda grad: grads.append(grad))
model.layer4[-1].register_forward_hook(capture)
for r in chosen:
    with zipfile.ZipFile('data/raw/ACRIMA_Database.zip') as z:
        with Image.open(io.BytesIO(z.read(manifest[r['image_id']]['member']))) as source:
            image=source.convert('RGB')
    x=transform(image).unsqueeze(0); acts.clear(); grads.clear(); model.zero_grad()
    logit=model(x)[0,0]
    target=logit if int(r['predicted_label'])==1 else -logit
    target.backward()
    a=acts[-1][0].detach(); g=grads[-1][0].detach()
    cam=torch.relu((g.mean((1,2))[:,None,None]*a).sum(0)); cam=cam-cam.min(); cam=cam/(cam.max()+1e-9)
    heat=Image.fromarray(np.uint8(cam.numpy()*255)).resize((224,224),Image.Resampling.BILINEAR)
    crop=ResNet18_Weights.IMAGENET1K_V1.transforms()(image) # display the same center-cropped view
    normalized=crop.numpy().transpose(1,2,0)
    rgb=np.uint8(np.clip((normalized*np.array([.229,.224,.225])+np.array([.485,.456,.406]))*255,0,255))
    view=Image.fromarray(rgb)
    overlay=Image.new('RGB',(224,224)); overlay.paste((255,0,0),mask=heat)
    merged=Image.blend(view,overlay,.4)
    panel=Image.new('RGB',(448,260),'white'); panel.paste(view,(0,0)); panel.paste(merged,(224,0))
    label=f"id {r['image_id']} | true {r['true_label']} | pred {r['predicted_label']} | p={float(r['probability_glaucoma']):.3f} | target={'glaucoma' if int(r['predicted_label']) else 'normal'}"
    ImageDraw.Draw(panel).text((8,232),label,fill='black')
    panel.save(out/f"{r['image_id']}.png")
print(f'Wrote {len(chosen)} Grad-CAM panels to {out}')


In [ ]:
%%writefile tools/render_pooled_gradcam.py
"""Render predicted-class Grad-CAM examples for four held-out datasets."""
import csv
import io
import json
import zipfile
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageDraw, ImageOps
from torchvision.models import resnet18
from torchvision.transforms import Normalize, ToTensor


torch.set_num_threads(4)
root = Path("outputs/pooled_four_dataset_frozen")
out = root / "gradcam"
out.mkdir(exist_ok=True)
sources = {
    "ACRIMA": ("acrima_test_predictions.csv", "data/raw/ACRIMA_Database.zip"),
    "ORIGA": ("origa_test_predictions.csv", "data/raw/ORIGA_candidate_figshare.zip"),
    "REFUGE": ("refuge_test_predictions.csv", "data/raw/ORIGA_candidate_figshare.zip"),
    "RIM-ONE r3": ("rim_one_r3_test_predictions.csv", "data/raw/RIM-ONE-r3.zip"),
}
model = resnet18(weights=None)
model.fc = torch.nn.Linear(512, 1)
model.load_state_dict(torch.load(root / "resnet18_classifier.pt", map_location="cpu", weights_only=True))
model.eval()
to_tensor = ToTensor()
normalize = Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
activations, gradients = [], []


def hook(_module, _inputs, output):
    activations.append(output)
    output.register_hook(lambda grad: gradients.append(grad))


model.layer4[-1].register_forward_hook(hook)
index = []
for dataset, (predictions_path, archive_path) in sources.items():
    rows = list(csv.DictReader((root / predictions_path).open(newline="", encoding="utf-8")))
    groups = {key: [] for key in ("TP", "TN", "FP", "FN")}
    for row in rows:
        y, p = int(row["true_label"]), int(row["predicted_label"])
        groups[("TP" if y else "TN") if y == p else ("FP" if p else "FN")].append(row)
    selected = [(key, group[0]) for key, group in groups.items() if group]
    folder = out / dataset.lower().replace(" ", "_").replace("-", "_")
    folder.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for outcome, row in selected:
            with Image.open(io.BytesIO(archive.read(row["member"]))) as source:
                image = source.convert("RGB")
                if dataset == "RIM-ONE r3":
                    image = image.crop((0, 0, image.width // 2, image.height))
                image = ImageOps.pad(image, (224, 224), method=Image.Resampling.BICUBIC, color=(0, 0, 0))
            tensor = normalize(to_tensor(image))[None]
            activations.clear()
            gradients.clear()
            model.zero_grad()
            logit = model(tensor)[0, 0]
            target = logit if int(row["predicted_label"]) else -logit
            target.backward()
            a = activations[-1][0].detach()
            g = gradients[-1][0].detach()
            cam = torch.relu((g.mean((1, 2))[:, None, None] * a).sum(0))
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-9)
            heat = Image.fromarray(np.uint8(cam.numpy() * 255)).resize((224, 224), Image.Resampling.BILINEAR)
            red = Image.new("RGB", (224, 224), (255, 30, 20))
            overlay = Image.blend(image, Image.composite(red, image, heat), 0.5)
            panel = Image.new("RGB", (448, 260), "white")
            panel.paste(image, (0, 0))
            panel.paste(overlay, (224, 0))
            label = f"{dataset} {outcome} | true={row['true_label']} pred={row['predicted_label']} p={float(row['probability_glaucoma']):.3f}"
            ImageDraw.Draw(panel).text((8, 233), label, fill="black")
            path = folder / (outcome + "_" + Path(row["member"]).stem + ".png")
            panel.save(path)
            index.append({"dataset": dataset, "outcome": outcome, "member": row["member"], "predicted_class_target": int(row["predicted_label"]), "panel": str(path).replace("\\", "/")})

(out / "index.json").write_text(json.dumps(index, indent=2), encoding="utf-8")
print(f"Wrote {len(index)} predicted-class Grad-CAM panels across {len(sources)} datasets to {out}")


In [ ]:
%%writefile tools/render_rim_predictions.py
"""Render held-out RIM-ONE images with true and predicted disc/cup masks."""
import csv
import io
import json
import sys
import zipfile
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageDraw
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from models.unet_small import UNet

torch.set_num_threads(4)
root=Path('outputs/rim_unet_baseline'); out=root/'examples'; out.mkdir(exist_ok=True)
scores=list(csv.DictReader((root/'test_predictions.csv').open(encoding='utf-8')))
scores.sort(key=lambda r:(float(r['disc_dice'])+float(r['cup_dice']))/2)
selected=[('worst',r) for r in scores[:4]]+[('best',r) for r in scores[-4:]]
manifest={r['image']:r for r in csv.DictReader(Path('data/audit/rim_one_r3_manifest.csv').open(encoding='utf-8'))}
model=UNet(); model.load_state_dict(torch.load(root/'best.pt',map_location='cpu',weights_only=True)); model.eval()
def overlay(image,disc,cup):
    base=np.asarray(image).copy(); base[disc]=.65*base[disc]+.35*np.array([0,220,100]); base[cup]=.55*base[cup]+.45*np.array([255,180,0])
    return Image.fromarray(base.astype('uint8'))
with zipfile.ZipFile('data/raw/RIM-ONE-r3.zip') as z, torch.no_grad():
    for quality, score in selected:
        name=score['image']
        row=manifest[name]
        image=Image.open(io.BytesIO(z.read(name))).convert('RGB'); image=image.crop((0,0,image.width//2,image.height)).resize((256,256))
        masks=[]
        for key in ('disc_mask','cup_mask'):
            mask=Image.open(io.BytesIO(z.read(row[key]))).convert('L'); mask=mask.crop((0,0,mask.width//2,mask.height)).resize((256,256),Image.Resampling.NEAREST)
            masks.append(np.asarray(mask)>127)
        x=torch.from_numpy(np.asarray(image).copy().transpose(2,0,1)).float().unsqueeze(0)/255
        pred=(model(x).sigmoid()[0].numpy()>.5)
        panel=Image.new('RGB',(768,280),'white'); panel.paste(image,(0,0)); panel.paste(overlay(image,*masks),(256,0)); panel.paste(overlay(image,*pred),(512,0))
        ImageDraw.Draw(panel).text((8,260),'Original',fill='black'); ImageDraw.Draw(panel).text((264,260),'Ground truth',fill='black'); ImageDraw.Draw(panel).text((520,260),f"Prediction D={float(score['disc_dice']):.2f} C={float(score['cup_dice']):.2f}",fill='black')
        panel.save(out/(quality+'_'+Path(name).stem+'.png'))
(out/'selection.json').write_text(json.dumps([{'quality':quality,**score} for quality,score in selected],indent=2),encoding='utf-8')
print(f'Wrote {len(selected)} best/worst held-out prediction panels to {out}')


In [ ]:
%%writefile tools/render_results.py
"""Render measured classification results for the later report and website."""
import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

base=Path('outputs/acrima_resnet18_frozen')
fig, axes=plt.subplots(2,2,figsize=(10,9),constrained_layout=True)
for col, (name, prediction_file, metrics_file) in enumerate((
    ('ACRIMA held-out','test_predictions.csv','metrics.json'),
    ('RIM-ONE external','rim_external_predictions.csv','rim_external_metrics.json'))):
    rows=list(csv.DictReader((base/prediction_file).open(encoding='utf-8')))
    y=np.array([int(r['true_label']) for r in rows]); p=np.array([float(r['probability_glaucoma']) for r in rows]); pred=np.array([int(r['predicted_label']) for r in rows])
    m=json.loads((base/metrics_file).read_text())
    cm=np.array([[m['TN'],m['FP']],[m['FN'],m['TP']]])
    ConfusionMatrixDisplay(cm,display_labels=['Normal','Glaucoma']).plot(ax=axes[0,col],colorbar=False,cmap='Blues')
    axes[0,col].set_title(f'{name}: confusion matrix (n={len(y)})')
    RocCurveDisplay.from_predictions(y,p,ax=axes[1,col],name=name)
    axes[1,col].plot([0,1],[0,1],'--',color='gray',linewidth=1)
    axes[1,col].set_title(f'{name}: ROC curve')
fig.suptitle('Frozen ResNet-18 feature baseline: internal vs external evaluation',fontsize=14)
fig.savefig(base/'comparison.png',dpi=160)
print(base/'comparison.png')


In [ ]:
%%writefile tools/render_baseline_architecture.py
"""Draw the two implemented baseline pipelines without implying a final model."""
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch


fig, ax = plt.subplots(figsize=(15, 7.2))
fig.patch.set_facecolor("#f8fbfc")
ax.set_facecolor("#f8fbfc")
ax.set_xlim(0, 15)
ax.set_ylim(0, 7.2)
ax.axis("off")


def box(x, y, w, h, title, detail, color):
    patch = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.12,rounding_size=0.12", linewidth=1.3, edgecolor=color, facecolor="white")
    ax.add_patch(patch)
    ax.text(x + w / 2, y + h * 0.63, title, ha="center", va="center", color="#173038", fontsize=11, weight="bold")
    ax.text(x + w / 2, y + h * 0.29, detail, ha="center", va="center", color="#53656d", fontsize=9)


def arrow(x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>", mutation_scale=15, linewidth=1.5, color="#7a9199"))


ax.text(0.25, 6.65, "Implemented glaucoma baselines", fontsize=19, weight="bold", color="#173038")
ax.text(0.25, 6.25, "Measured on ACRIMA and RIM-ONE r3; final five-dataset comparison is pending", fontsize=10, color="#53656d")

ax.text(0.25, 5.72, "CLASSIFICATION + EXPLANATION", fontsize=11, weight="bold", color="#147d86")
classification = [
    (0.3, "ACRIMA fundus", "RGB image"),
    (3.25, "Preprocess", "resize 256; crop 224"),
    (6.2, "ResNet-18", "ImageNet weights; frozen"),
    (9.15, "Logistic head", "512 features; balanced"),
    (12.1, "Prediction + XAI", "probability; Grad-CAM"),
]
for x, title, detail in classification:
    box(x, 4.45, 2.55, 0.95, title, detail, "#147d86")
for x in (2.85, 5.8, 8.75, 11.7):
    arrow(x, 4.93, x + 0.4, 4.93)
ax.text(12.1, 4.15, "Grad-CAM uses ResNet layer 4", fontsize=8.5, color="#53656d")

ax.text(0.25, 3.5, "OPTIC DISC + CUP SEGMENTATION", fontsize=11, weight="bold", color="#a46425")
segmentation = [
    (0.3, "RIM-ONE stereo", "left view only"),
    (3.25, "Preprocess", "RGB + masks at 256²"),
    (6.2, "Compact U-Net", "encoder 16–32–64–128"),
    (9.15, "Two-channel head", "disc and cup logits"),
    (12.1, "Evaluation", "0.5 mask; Dice + IoU"),
]
for x, title, detail in segmentation:
    box(x, 2.25, 2.55, 0.95, title, detail, "#a46425")
for x in (2.85, 5.8, 8.75, 11.7):
    arrow(x, 2.73, x + 0.4, 2.73)
ax.text(6.2, 1.87, "U-Net skip connections pass encoder features to decoder stages", fontsize=8.5, color="#53656d")

ax.text(0.3, 0.95, "Protocol: fixed train / validation / test splits • test data used only for final evaluation • ACRIMA has no segmentation masks", fontsize=10, color="#384e56")
destination = Path("outputs/architecture_baselines.png")
destination.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(destination, dpi=170, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close(fig)
print(destination)


In [ ]:
if RUN_VISUALS_AND_EXPORT:
    for name in ('render_gradcam.py', 'render_pooled_gradcam.py', 'render_rim_predictions.py', 'render_results.py', 'render_baseline_architecture.py'):
        run_script(name)
for path in ('outputs/architecture_baselines.png', 'outputs/acrima_resnet18_frozen/comparison.png'):
    display(Image.open(ROOT / path))
print('Step 8 complete: visual explanations and architecture figure saved.')

## Step 9 — Export measured results for the later website

The export includes only numbers produced by this run. It preserves per-image predictions, model/dataset/split keys, comparison data, and paths to local visual assets. Never fill missing combinations with invented metrics.

Run the source cells below, then the stage runner. Every file they create comes from code **inside this notebook**.

In [ ]:
%%writefile tools/export_results.py
"""Export only completed, measured experiments for the future website."""
import csv
import json
from pathlib import Path

root=Path('outputs')
def read_json(path): return json.loads(path.read_text(encoding='utf-8'))
def read_csv(path): return list(csv.DictReader(path.open(encoding='utf-8')))
experiments=[]
acr=root/'acrima_resnet18_frozen'
experiments.append({'id':'acrima_resnet18_frozen_internal','task':'classification','dataset':'ACRIMA','split':'test','metrics':read_json(acr/'metrics.json'),'predictions':read_csv(acr/'test_predictions.csv'),'gradcam_panels':[str(p).replace('\\','/') for p in sorted((acr/'gradcam').glob('*.png'))],'source':'outputs/acrima_resnet18_frozen'})
experiments.append({'id':'acrima_resnet18_frozen_rim_external','task':'classification','dataset':'RIM-ONE r3','split':'external','metrics':read_json(acr/'rim_external_metrics.json'),'predictions':read_csv(acr/'rim_external_predictions.csv'),'source':'outputs/acrima_resnet18_frozen'})
experiments.append({'id':'acrima_resnet18_frozen_origa_external','task':'classification','dataset':'ORIGA','split':'external_all_650','metrics':read_json(acr/'origa_external_metrics.json'),'predictions':read_csv(acr/'origa_external_predictions.csv'),'source':'outputs/acrima_resnet18_frozen'})
experiments.append({'id':'acrima_resnet18_frozen_refuge_validation_external','task':'classification','dataset':'REFUGE','split':'official_validation','metrics':read_json(acr/'refuge_val_external_metrics.json'),'predictions':read_csv(acr/'refuge_val_external_predictions.csv'),'source':'outputs/acrima_resnet18_frozen'})
rim=root/'rim_unet_baseline'
experiments.append({'id':'rim_unet_baseline','task':'segmentation','dataset':'RIM-ONE r3','split':'test','metrics':read_json(rim/'metrics.json'),'predictions':read_csv(rim/'test_predictions.csv'),'example_panels':[str(p).replace('\\','/') for p in sorted((rim/'examples').glob('*.png'))],'source':'outputs/rim_unet_baseline'})
pooled=root/'pooled_four_dataset_frozen'
pooled_metrics=read_json(pooled/'metrics.json')
for dataset,stem,split in (('ACRIMA','acrima','test'),('ORIGA','origa','provided_B_test'),('REFUGE','refuge','official_validation'),('RIM-ONE r3','rim_one_r3','fixed_test_non_suspect')):
    experiments.append({'id':'pooled_four_frozen_'+stem,'task':'classification','dataset':dataset,'split':split,'model_protocol':pooled_metrics['protocol'],'metrics':pooled_metrics['test_metrics'][dataset],'predictions':read_csv(pooled/(stem+'_test_predictions.csv')),'gradcam_panels':[str(p).replace('\\','/') for p in sorted((pooled/'gradcam'/stem).glob('*.png'))],'source':'outputs/pooled_four_dataset_frozen'})
experiments.append({'id':'pooled_four_frozen_refuge_official_test','task':'classification','dataset':'REFUGE','split':'official_test','metrics':read_json(pooled/'refuge_official_test_metrics.json'),'predictions':read_csv(pooled/'refuge_official_test_predictions.csv'),'source':'outputs/pooled_four_dataset_frozen'})
crop=root/'rim_whole_vs_crop'
if (crop/'metrics.json').exists():
    crop_metrics=read_json(crop/'metrics.json')
    for variant in ('whole','crop'):
        experiments.append({'id':'rim_matched_'+variant+'_frozen','task':'classification','dataset':'RIM-ONE r3','split':'fixed_test_non_suspect','metrics':{k:v for k,v in crop_metrics[variant].items() if k not in ('val_trials',)},'predictions':read_csv(crop/(variant+'_test_predictions.csv')),'protocol':crop_metrics['protocol'][variant],'source':'outputs/rim_whole_vs_crop'})
pooled_seg=root/'pooled_segmentation'
if (pooled_seg/'metrics.json').exists():
    pooled_seg_metrics=read_json(pooled_seg/'metrics.json')
    pooled_seg_predictions=read_csv(pooled_seg/'test_predictions.csv') if (pooled_seg/'test_predictions.csv').exists() else []
    for record in pooled_seg_metrics['test_metrics']:
        dataset=record['dataset']
        dataset_predictions=[row for row in pooled_seg_predictions if row['dataset']==dataset]
        experiments.append({'id':'pooled_segmentation_'+dataset.lower().replace(' ','_').replace('-','_'),'task':'segmentation','dataset':dataset,'split':'fixed_test','metrics':record,'predictions':dataset_predictions,'example_panels':[str(p).replace('\\','/') for p in sorted((pooled_seg/'examples').glob(f"*{dataset.lower().replace(' ','_').replace('-','_')}*.png"))],'source':'outputs/pooled_segmentation'})
drishti_seg=root/'drishti_gs1_mirror'
drishti_clf=root/'drishti_gs1_mirror_classifier'
if (drishti_seg/'metrics.json').exists():
    drishti_metrics=read_json(drishti_seg/'metrics.json')
    drishti_predictions=read_csv(drishti_seg/'test_predictions.csv') if (drishti_seg/'test_predictions.csv').exists() else []
    experiments.append({'id':'drishti_gs1_mirror_segmentation','task':'segmentation','dataset':'DRISHTI-GS1','split':'mirror_official_style_test','source':'lokeshsaipureddi Kaggle mirror','metrics':drishti_metrics,'predictions':drishti_predictions,'example_panels':[str(p).replace('\\','/') for p in sorted((drishti_seg/'examples').glob('*.png'))],'source':'outputs/drishti_gs1_mirror'})
if (drishti_clf/'metrics.json').exists():
    drishti_clf_metrics=read_json(drishti_clf/'metrics.json')
    experiments.append({'id':'drishti_gs1_mirror_frozen_classifier','task':'classification','dataset':'DRISHTI-GS1','split':'mirror_train_val_test_40_10_51','source':'lokeshsaipureddi Kaggle mirror','metrics':drishti_clf_metrics,'predictions':read_csv(drishti_clf/'test_predictions.csv'),'source':'outputs/drishti_gs1_mirror_classifier'})
finetuned=root/'pooled_finetuned_classifier'
if (finetuned/'metrics.json').exists():
    finetuned_metrics=read_json(finetuned/'metrics.json')
    finetuned_predictions=read_csv(finetuned/'test_predictions.csv') if (finetuned/'test_predictions.csv').exists() else []
    for dataset, metrics in finetuned_metrics['test_metrics'].items():
        if dataset == 'REFUGE':
            continue
        experiments.append({'id':'pooled_finetuned_'+dataset.lower().replace(' ','_').replace('-','_'),'task':'classification','dataset':dataset,'split':'fixed_test','source':'outputs/pooled_finetuned_classifier','metrics':metrics,'predictions':[row for row in finetuned_predictions if row['dataset']==dataset],'protocol':finetuned_metrics['protocol']})
    for source_split, split_name in (('val', 'official_validation'), ('test', 'official_test')):
        experiments.append({'id':'pooled_finetuned_refuge_'+source_split,'task':'classification','dataset':'REFUGE','split':split_name,'source':'outputs/pooled_finetuned_classifier','metrics':finetuned_metrics['refuge_test_metrics_by_source_split'][source_split],'predictions':[row for row in finetuned_predictions if row['dataset']=='REFUGE' and row['source_split']==source_split],'protocol':finetuned_metrics['protocol']})
payload={'schema_version':1,'status':'partial_experiments','note':'Only completed measured experiments are listed. DRISHTI-GS1 results use a third-party Kaggle mirror and are not official-source verified.','assets':{'baseline_architecture':'outputs/architecture_baselines.png','classification_comparison':'outputs/acrima_resnet18_frozen/comparison.png','same_split_classifier_comparison':'outputs/classifier_common_split_comparison.png','rim_whole_vs_crop_comparison':'outputs/rim_whole_vs_crop/comparison.png'},'same_split_comparison':read_json(root/'classifier_common_split_comparison.json'),'rim_whole_vs_crop_comparison':read_json(crop/'metrics.json'),'pooled_segmentation':read_json(pooled_seg/'metrics.json') if (pooled_seg/'metrics.json').exists() else None,'drishti_gs1_mirror':{'segmentation':read_json(drishti_seg/'metrics.json') if (drishti_seg/'metrics.json').exists() else None,'classification':read_json(drishti_clf/'metrics.json') if (drishti_clf/'metrics.json').exists() else None},'experiments':experiments}
dest=root/'website_results_partial.json'; dest.write_text(json.dumps(payload,indent=2),encoding='utf-8')
print(dest)


In [ ]:
%%writefile tools/validate_website_export.py
"""Reject missing artifacts or invented combinations in the partial site export."""
import json
from pathlib import Path


root = Path("outputs")
payload = json.loads((root / "website_results_partial.json").read_text(encoding="utf-8"))
assert payload["status"] == "partial_experiments"
assert len(payload["experiments"]) >= 17
ids = [experiment["id"] for experiment in payload["experiments"]]
assert len(set(ids)) == len(ids)
for experiment in payload["experiments"]:
    assert experiment["dataset"] in ("ACRIMA", "RIM-ONE r3", "ORIGA", "REFUGE", "DRISHTI-GS1")
    assert not (experiment["dataset"] == "ACRIMA" and experiment["task"] == "segmentation")
    assert len(experiment["predictions"]) == experiment["metrics"]["n_test"]
    for field in ("gradcam_panels", "example_panels"):
        for path in experiment.get(field, []):
            assert Path(path).is_file(), path
for path in payload["assets"].values():
    assert Path(path).is_file(), path
print("Validated", len(ids), "measured experiment combinations:", ", ".join(ids))


In [ ]:
%%writefile website/index.html
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Glaucoma MLDL - Measured Results</title>
  <link rel="stylesheet" href="./styles.css">
</head>
<body>
  <header class="hero">
    <p class="eyebrow">MEASURED PARTIAL CHECKPOINT</p>
    <h1>Glaucoma classification and segmentation</h1>
    <p>Frozen baselines, fixed splits, per-image predictions, and saved explainability examples.</p>
  </header>
  <main>
    <section class="notice">
      <strong>Partial results only.</strong>
      DRISHTI-GS1 is not included. These cards come exclusively from saved measured artifacts.
    </section>
    <section id="summary" class="summary"></section>
    <section>
      <div class="section-heading">
        <h2>Experiment record</h2>
        <label>Dataset
          <select id="dataset-filter">
            <option value="all">All datasets</option>
          </select>
        </label>
      </div>
      <div id="experiments" class="cards"></div>
    </section>
  </main>
  <script src="./app.js"></script>
</body>
</html>


In [ ]:
%%writefile website/styles.css
:root {
  --ink: #172033;
  --muted: #657187;
  --paper: #f4f7fb;
  --card: #ffffff;
  --line: #dbe3ee;
  --accent: #3157d5;
  --accent-soft: #edf1ff;
  --warning: #fff4dc;
  --warning-line: #d8b55a;
}
* { box-sizing: border-box; }
body {
  margin: 0;
  font-family: Inter, Aptos, "Segoe UI", sans-serif;
  color: var(--ink);
  background: var(--paper);
}
.hero {
  padding: 64px max(24px, 8vw) 48px;
  background: linear-gradient(135deg, #172033 0%, #253c7c 100%);
  color: white;
}
.hero h1 { margin: 8px 0 12px; font-size: clamp(38px, 6vw, 72px); line-height: 0.95; max-width: 900px; }
.hero p { max-width: 760px; line-height: 1.6; }
.eyebrow { letter-spacing: 0.16em; font-size: 12px; font-weight: 800; color: #b9c8ff; }
main { max-width: 1180px; margin: 0 auto; padding: 32px 24px 64px; }
.notice { border: 1px solid var(--warning-line); background: var(--warning); border-radius: 12px; padding: 16px 18px; }
.summary { display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin: 24px 0; }
.summary-card { background: var(--card); border: 1px solid var(--line); border-radius: 14px; padding: 18px; }
.summary-card strong { display: block; font-size: 28px; }
.summary-card span { color: var(--muted); font-size: 13px; }
.section-heading { display: flex; align-items: end; justify-content: space-between; gap: 16px; }
.section-heading label { color: var(--muted); font-size: 13px; }
select { display: block; margin-top: 6px; border: 1px solid var(--line); border-radius: 8px; padding: 8px 34px 8px 10px; background: white; }
.cards { display: grid; grid-template-columns: repeat(2, 1fr); gap: 16px; margin-top: 18px; }
.card { background: var(--card); border: 1px solid var(--line); border-radius: 14px; padding: 20px; }
.card h3 { margin: 0 0 6px; }
.tag { display: inline-block; margin-right: 6px; border-radius: 999px; background: var(--accent-soft); color: #25429c; padding: 4px 8px; font-size: 12px; }
.metrics { display: grid; grid-template-columns: repeat(4, 1fr); gap: 8px; margin-top: 16px; }
.metric { background: #f8fafe; border-radius: 8px; padding: 10px; }
.metric strong { display: block; font-size: 17px; }
.metric span { color: var(--muted); font-size: 11px; text-transform: uppercase; }
@media (max-width: 760px) {
  .summary, .cards { grid-template-columns: 1fr; }
  .metrics { grid-template-columns: repeat(2, 1fr); }
  .section-heading { align-items: start; flex-direction: column; }
}


In [ ]:
%%writefile website/app.js
const metricKeys = [
  "accuracy",
  "auc",
  "f1",
  "precision",
  "recall",
  "test_disc_dice",
  "test_disc_iou",
  "test_cup_dice",
  "test_cup_iou"
];

fetch("../results/website_results_partial.json")
  .then(response => {
    if (!response.ok) throw new Error("Could not load measured export");
    return response.json();
  })
  .then(payload => render(payload))
  .catch(error => {
    document.getElementById("experiments").innerHTML =
      `<article class="card"><h3>Export unavailable</h3><p>${error.message}</p></article>`;
  });

function render(payload) {
  const experiments = payload.experiments;
  const datasets = [...new Set(experiments.map(item => item.dataset))].sort();
  const filter = document.getElementById("dataset-filter");
  datasets.forEach(dataset => {
    const option = document.createElement("option");
    option.value = dataset;
    option.textContent = dataset;
    filter.appendChild(option);
  });

  document.getElementById("summary").innerHTML = [
    [String(experiments.length), "measured combinations"],
    [String(datasets.length), "datasets represented"],
    ["705", "ACRIMA images"],
    ["159", "RIM-ONE images"]
  ].map(([value, label]) =>
    `<article class="summary-card"><strong>${value}</strong><span>${label}</span></article>`
  ).join("");

  function draw() {
    const selected = filter.value;
    const visible = selected === "all"
      ? experiments
      : experiments.filter(item => item.dataset === selected);
    document.getElementById("experiments").innerHTML = visible.map(card).join("");
  }
  filter.addEventListener("change", draw);
  draw();
}

function card(experiment) {
  const metrics = experiment.metrics || {};
  const values = [];
  for (const key of metricKeys) {
    if (typeof metrics[key] === "number") {
      values.push([key, metrics[key]]);
    }
  }
  if (metrics.test_disc_dice != null) values.push(["disc Dice", metrics.test_disc_dice]);
  if (metrics.test_cup_dice != null) values.push(["cup Dice", metrics.test_cup_dice]);
  const metricHtml = values.slice(0, 4).map(([key, value]) =>
    `<div class="metric"><strong>${Number(value).toFixed(4)}</strong><span>${key}</span></div>`
  ).join("");
  return `<article class="card">
    <h3>${experiment.dataset}</h3>
    <span class="tag">${experiment.task}</span>
    <span class="tag">${experiment.split}</span>
    <div class="metrics">${metricHtml}</div>
  </article>`;
}


In [ ]:
if RUN_VISUALS_AND_EXPORT:
    run_script('export_results.py')
    run_script('validate_website_export.py')
    shutil.copy2(ROOT / 'outputs/website_results_partial.json', ROOT / 'results/website_results_partial.json')
payload = json.loads((ROOT / 'results/website_results_partial.json').read_text())
print('Measured combinations:', len(payload['experiments']))
print('Datasets represented:', sorted({row['dataset'] for row in payload['experiments']}))
print('Step 9 complete: website JSON written to results/website_results_partial.json.')

## Step 10 — Make a report draft and downloadable package

This report text is generated from the metrics saved in **this run**. Review it with your team and make the final document 2–3 pages. The ZIP contains code outputs, metrics, plots, and trained checkpoints; it excludes raw dataset archives and DRISHTI mirror images. Keep raw images private under their source terms.

In [ ]:
# Generate a concise report draft from measured files, then package submission artifacts.
pooled = json.loads((ROOT / 'outputs/pooled_four_dataset_frozen/metrics.json').read_text())
seg = json.loads((ROOT / 'outputs/pooled_segmentation/metrics.json').read_text())
fine = json.loads((ROOT / 'outputs/pooled_finetuned_classifier/metrics.json').read_text())
official = json.loads((ROOT / 'outputs/pooled_four_dataset_frozen/refuge_official_test_metrics.json').read_text())
lines = ['# Glaucoma Image Classification, Disc/Cup Segmentation and Explainable AI', '',
         '## Introduction', 'We evaluated retinal fundus models across ACRIMA, RIM-ONE r3, ORIGA, REFUGE, and a DRISHTI-GS1 Kaggle mirror. The DRISHTI results are mirror-based and must be labeled as such.', '',
         '## Methodology', 'Fixed training, validation and test assignments were recorded before model fitting. ACRIMA has classification labels only. Segmentation predicts optic disc and cup; classifiers use full images or predicted disc crops. Test masks were not used to crop classification images. The source archives and masks were audited before training.', '',
         '## Architecture', 'A frozen or fine-tuned ResNet-18 predicts glaucoma; a two-channel U-Net segments disc and cup. The architecture diagram is saved at outputs/architecture_baselines.png. Grad-CAM targets the predicted class.', '',
         '## Results', '| Dataset | Pooled frozen AUC | Fine-tuned AUC |', '|---|---:|---:|']
for dataset, old in pooled['test_metrics'].items():
    new = fine['refuge_test_metrics_by_source_split']['val'] if dataset == 'REFUGE' else fine['test_metrics'].get(dataset, {})
    lines.append(f"| {dataset} | {old['auc']:.3f} | {new.get('auc', float('nan')):.3f} |")
lines += ['', f"Independent REFUGE official-test AUC for the pooled frozen model: {official['auc']:.3f} (n={official['n_test']}).", '',
          f"Independent REFUGE official-test AUC for the fine-tuned model: {fine['refuge_test_metrics_by_source_split']['test']['auc']:.3f} (n={fine['refuge_test_metrics_by_source_split']['test']['n_test']}).", '',
          '### Segmentation', '| Dataset | Disc Dice | Cup Dice |', '|---|---:|---:|']
for row in seg['test_metrics']:
    lines.append(f"| {row['dataset']} | {row['test_disc_dice']:.3f} | {row['test_cup_dice']:.3f} |")
lines += ['', '## Conclusion', 'Compare whole-image and predicted-disc-crop performance on the same held-out images. Discuss domain shift, class imbalance, the small RIM-ONE and DRISHTI samples, and third-party DRISHTI mirror provenance. These research models are not clinical devices.', '']
(ROOT / 'report/glaucoma_project_report_generated.md').write_text('\n'.join(lines), encoding='utf-8')
package = ROOT / 'glaucoma_submission_artifacts.zip'
with zipfile.ZipFile(package, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for folder in ('outputs', 'results', 'report', 'website'):
        for path in (ROOT / folder).rglob('*'):
            if path.is_file(): z.write(path, path.relative_to(ROOT))
print('Report:', ROOT / 'report/glaucoma_project_report_generated.md')
print('Artifact ZIP:', package, '| MiB:', round(package.stat().st_size / 2**20, 1))
print('Everything above was generated from this notebook. Dataset archives remain outside the package.')

### Optional: download the artifact ZIP from Colab

This is an **output download** after training, not a dataset upload. If the ZIP is large, copy it to your free Google Drive instead. The raw dataset archives are not included.

In [ ]:
# Uncomment only when you want the artifact ZIP on your computer.
# from google.colab import files
# files.download(str(ROOT / 'glaucoma_submission_artifacts.zip'))